In [ ]:
# Create Dataset

import torch
from torch.utils.data import Dataset
import numpy as np
import os
import cv2
import json
import random
import pandas as pd
from scipy.ndimage import rotate

class Early2LateWithLatentDataset(Dataset):
    def __init__(self, pet_dir, dataset_type="train", fold=0, 
                 split=None, resize_image_size=(128, 128), 
                 json_path="/disk/disk4/allen/SAM2-UNet-main/subjects_data_with_abeta.json", 
                 stats_csv="/disk/disk4/allen/SAM2-UNet-main/subject_stats_Early2Late_withLatent_FULL_0729.csv"
                 #stats_csv= "/disk/disk4/allen/SAM2-UNet-main/subject_stats_Early2Late_withLatent_FULL_06274+1m.csv"
                 ):
        super().__init__()
        self.pet_dir = pet_dir
        self.dataset_type = dataset_type
        self.fold = fold
        self.resize_image_size = resize_image_size
        self.split = split or {}

        with open(json_path, "r") as f:
            self.subject_metadata = json.load(f)

        self.stats = self._load_stats(stats_csv)

        if str(fold) not in self.split:
            raise ValueError(f"Fold {fold} not found in split")

        val_index = (fold + len(self.split) - 3) % len(self.split)
        test_index = (fold + len(self.split) - 2) % len(self.split)
        self.val_subjects = set(self.split[str(val_index)])
        self.test_subjects = set(self.split[str(test_index)])

        self.data_files = self._prepare_files()

    def _load_stats(self, stats_csv):
        df = pd.read_csv(stats_csv)
        stats = {}
        for _, row in df.iterrows():
            stats[(row["Type"], row["Subject"])] = {
                "min": row["Min Value"],
                "max": row["Max Value"]
            }
        return stats

    def _normalize(self, data, subject, reference_key="data"):
        key = (reference_key, subject)
        if key not in self.stats:
            return data
        stat = self.stats[key]
        min_val, max_val = stat["min"], stat["max"]
        if max_val - min_val <= 1e-5:
            return np.zeros_like(data)
        return np.clip((data - min_val) / (max_val - min_val), 0, 1)

    def _prepare_files(self):
        files = []
        for group_num, subjects in self.split.items():
            data_dir = os.path.join(self.pet_dir, f"group{group_num}", "data")
            if not os.path.exists(data_dir):
                continue
            for fname in os.listdir(data_dir):
                subject = fname.split('_')[0]
                if self.dataset_type == "train" and subject not in self.val_subjects and subject not in self.test_subjects:
                    files.append((group_num, fname))
                elif self.dataset_type == "val" and subject in self.val_subjects:
                    files.append((group_num, fname))
                elif self.dataset_type == "test" and subject in self.test_subjects:
                    files.append((group_num, fname))
        return files

    def __len__(self):
        return len(self.data_files)

    def _rotate_small_angle(self, arr, angle):
        return rotate(arr, angle=angle, reshape=False, order=1, mode='reflect')

    def _augment(self, early, gt, latent):
        if random.random() < 0.3:
            angle = random.choice([-30, 30])
            early = self._rotate_small_angle(early, angle)
            gt = self._rotate_small_angle(gt, angle)
            latent = np.stack([self._rotate_small_angle(latent[..., i], angle) for i in range(latent.shape[-1])], axis=-1)
        return early, gt, latent

    def __getitem__(self, idx):
        group_num, fname = self.data_files[idx]
        subject = fname.split('_')[0]
        slice_idx = int(fname.split('_')[-1].split('.')[0])

        base_path = os.path.join(self.pet_dir, f"group{group_num}")
        early_path = os.path.join(base_path, "data", fname)
        gt_path = os.path.join(base_path, "ground_truth", fname)
        latent_path = os.path.join(base_path, "latent_target", fname)

        early = np.load(early_path)
        gt = np.load(gt_path)
        latent = np.load(latent_path)

        early = cv2.resize(early.squeeze(), self.resize_image_size)[..., np.newaxis]
        gt = cv2.resize(gt, self.resize_image_size)

        if latent.ndim == 2:
            latent = latent[..., np.newaxis]

        latent = np.stack([cv2.resize(latent[..., i], self.resize_image_size) for i in range(latent.shape[-1])], axis=-1)

        if self.dataset_type == "train":
            early, gt, latent = self._augment(early, gt, latent)

        early = self._normalize(early, subject, "data")
        gt = self._normalize(gt, subject, "data")
        latent = np.stack([self._normalize(latent[..., i], subject, "data") for i in range(latent.shape[-1])], axis=-1)

        # === 轉換為 torch tensor 並補上 batch 維度 ===
        early = torch.tensor(early.transpose(2, 0, 1).copy(), dtype=torch.float32).unsqueeze(0)   # [1, 1, H, W]
        gt = torch.tensor(gt.copy(), dtype=torch.float32).unsqueeze(0).unsqueeze(0)               # [1, 1, H, W]
        latent = torch.tensor(latent.copy(), dtype=torch.float32)                                 # [H, W, M]
        latent = latent.permute(2, 0, 1).unsqueeze(1).unsqueeze(0)                                # [1, M, 1, H, W]

        # === DEBUG: 列印所有 tensor 的 shape ===
        #print(f"[DEBUG] early.shape         = {early.shape}")         # [1, 1, H, W]
        #print(f"[DEBUG] ground_truth.shape  = {gt.shape}")            # [1, 1, H, W]
        #print(f"[DEBUG] latent_target.shape = {latent.shape}")        # [1, M, 1, H, W]

        metadata = self.subject_metadata.get(subject.replace("Subject", "patient"), {})
        abeta_value = metadata.get("Amyloid beta", -1)
        abeta_value = int(abeta_value) if isinstance(abeta_value, (int, float)) and not pd.isna(abeta_value) else -1

        return {
            "input": early,
            "ground_truth": gt,
            "latent_target": latent,
            "subject": subject,
            "slice": slice_idx,
            "diagnosis": metadata.get("Diagnosis", "Unknown"),
            "instrument": metadata.get("Instrument", "Unknown"),
            "abeta": abeta_value
        }



In [ ]:
# Dataset Visual
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import json
import numpy as np

# 讀取 split_all (5-fold)
with open("stratified_5fold_all.json", "r") as f:
    split_all = json.load(f)

# 建立 dataset
dataset = Early2LateWithLatentDataset(
    pet_dir= "/disk/disk4/allen/SAM2-UNet-main/Preprocessed_Early2Late_withLatent_0729",
    dataset_type="val",
    fold=3,
    split=split_all,
    resize_image_size=(128, 128),
    json_path="/disk/disk4/allen/SAM2-UNet-main/subjects_data_with_abeta.json"
)

# 建立 DataLoader
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# 取得一筆資料
for batch in dataloader:
    input_tensor = batch["input"].squeeze().numpy()             # [256, 256]
    gt_tensor = batch["ground_truth"].squeeze().numpy()         # [256, 256]
    latent_tensor = batch["latent_target"].squeeze().numpy()    # [3, 256, 256]
    M = latent_tensor.shape[0]

    # 🔍 顯示 metadata
    print("🔹 subject     :", batch["subject"][0])
    print("🔹 slice       :", batch["slice"][0].item())
    print("🔹 diagnosis   :", batch["diagnosis"][0])
    print("🔹 instrument  :", batch["instrument"][0])
    print("🔹 Aβ          :", batch["abeta"][0].item())

    print("\n--- Value Range ---")
    print(f"Input  : min={input_tensor.min():.4f}, max={input_tensor.max():.4f}")
    print(f"GT     : min={gt_tensor.min():.4f}, max={gt_tensor.max():.4f}")
    for i in range(M):
        print(f"Mid {i}: min={latent_tensor[i].min():.4f}, max={latent_tensor[i].max():.4f}")

    # 顯示影像（Early + Mid1~3 + GT）
    fig, axs = plt.subplots(1, 5, figsize=(20, 4))

    axs[0].imshow(np.rot90(input_tensor), cmap='hot')
    axs[0].set_title('EarlyFrame')

    for i in range(3):
        if i < M:
            axs[i+1].imshow(np.rot90(latent_tensor[i]), cmap='hot')
            axs[i+1].set_title(f'Mid{i+1}')
        else:
            axs[i+1].imshow(np.zeros_like(input_tensor), cmap='gray')
            axs[i+1].set_title(f'Mid{i+1} (Missing)')

    axs[4].imshow(np.rot90(gt_tensor), cmap='hot')
    axs[4].set_title('LateFrame')

    for ax in axs:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    break


In [ ]:
# Check Train Test Valid # of Slice
import os, json, collections
from pathlib import Path

ROOT = Path("Preprocessed_Early2Late_withLatent_0729")

def count_raw_slices(root: Path):
    total = 0
    for group_dir in root.glob("group*"):
        data_dir = group_dir / "data"
        if data_dir.is_dir():
            total += len(list(data_dir.glob("*.npy")))
    return total

def count_by_split(root: Path, split_json="stratified_5fold_all.json", fold=0):
    with open(split_json) as f:
        split_all = json.load(f)

    val_idx  = (fold + len(split_all) - 3) % len(split_all)
    test_idx = (fold + len(split_all) - 2) % len(split_all)

    val_set  = set(split_all[str(val_idx)])
    test_set = set(split_all[str(test_idx)])

    counts = collections.Counter(train=0, val=0, test=0)

    for group_dir in root.glob("group*"):
        data_dir = group_dir / "data"
        if not data_dir.is_dir():
            continue
        for fn in data_dir.glob("*.npy"):
            subject = fn.stem.split('_')[0]          # e.g. Subject007
            if subject in val_set:
                counts['val']  += 1
            elif subject in test_set:
                counts['test'] += 1
            else:
                counts['train'] += 1
    return counts

if __name__ == "__main__":
    raw_total = count_raw_slices(ROOT)
    split_cnt = count_by_split(ROOT, fold=3)

    print(f"📊 Raw slices (all groups): {raw_total}")
    print(f"    • Train : {split_cnt['train']}")
    print(f"    • Val   : {split_cnt['val']}")
    print(f"    • Test  : {split_cnt['test']}")

    # 估計經過隨機旋轉後「看到」的期望樣本數（僅 train）
    aug_train = split_cnt['train'] * 1.3      # 因為 p=0.3 進行一次旋轉
    print(f"≈ Expected distinct train images after on-the-fly augmentation: {aug_train:.0f}")


In [ ]:
# LKMUNet Backbone + SDEFunc & SDEBlock
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchsde
from typing import Union, Type, List, Tuple

####################################
# 1. 先定義 SDEFunc 與 SDEBlock
####################################
class SDEFunc(nn.Module):
    def __init__(self, dim, noise_type="diagonal", sde_type="ito"):
        super().__init__()
        self.dim = dim
        self.noise_type = noise_type
        self.sde_type = sde_type

        self.sde_mu = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, 3, 1, 1)
        )
        self.sde_sigma = nn.Conv2d(dim, dim, kernel_size=1)

        # === FiLM 參數初始化為 None
        self.gamma = None
        self.beta = None

    def set_film_params(self, gamma: torch.Tensor, beta: torch.Tensor):
        """
        設定 FiLM modulation 向量，用於調變 mu(x) 的輸出。
        gamma, beta: shape = [B, C, 1, 1]（與 feature map 對應通道）
        """
        self.gamma = gamma
        self.beta = beta

    def f(self, t, x):
        B, D = x.shape
        spatial_size = int((D // self.dim) ** 0.5)
        x_map = x.view(B, self.dim, spatial_size, spatial_size)  # [B, C, H, W]

        out = self.sde_mu(x_map)  # [B, C, H, W]

        # === FiLM modulation
        if self.gamma is not None and self.beta is not None:
            out = self.gamma * out + self.beta

        return out.view(B, -1)

    def g(self, t, x):
        B, D = x.shape
        spatial_size = int((D // self.dim) ** 0.5)
        x_map = x.view(B, self.dim, spatial_size, spatial_size)
        return self.sde_sigma(x_map).view(B, -1)

class SDEBlock(nn.Module):
    def __init__(self, sdefunc, tolerance=1e-3, adjoint=False, steps=10):
        super().__init__()
        self.sdefunc = sdefunc
        self.tolerance = tolerance
        self.adjoint = adjoint
        self.steps = steps

    def forward(self, x, integration_time=None, gamma=None, beta=None):
        B, C, H, W = x.shape
        x_vec = x.reshape(B, -1)  # ✅ 安全替代 view

        if gamma is not None and beta is not None:
            # 設定 FiLM 調變參數
            self.sdefunc.set_film_params(gamma, beta)

        sde_int = torchsde.sdeint_adjoint if self.adjoint else torchsde.sdeint

        if integration_time is None:
            integration_time = torch.linspace(0, 1, self.steps + 1, device=x.device)

        out = sde_int(
            self.sdefunc, x_vec,
            integration_time,
            dt=1.0 / self.steps,
            method='euler',
            rtol=self.tolerance,
            atol=self.tolerance
        )

        out_all = out.reshape(out.shape[0], B, C, H, W)  # [T, B, D] → [T, B, C, H, W]
        return out_all

from re import S
from xml.dom import xmlbuilder
import numpy as np
import torch
from torch import nn
from typing import Union, Type, List, Tuple

from dynamic_network_architectures.building_blocks.helper import get_matching_convtransp
from dynamic_network_architectures.building_blocks.plain_conv_encoder import PlainConvEncoder

from dynamic_network_architectures.building_blocks.simple_conv_blocks import StackedConvBlocks
from dynamic_network_architectures.building_blocks.residual import StackedResidualBlocks

from dynamic_network_architectures.building_blocks.unet_decoder import UNetDecoder

from dynamic_network_architectures.building_blocks.helper import maybe_convert_scalar_to_list, get_matching_pool_op
from dynamic_network_architectures.building_blocks.residual import BasicBlockD, BottleneckD
from torch.nn.modules.conv import _ConvNd
from torch.nn.modules.dropout import _DropoutNd
from torch.cuda.amp import autocast
from dynamic_network_architectures.building_blocks.helper import convert_conv_op_to_dim
from nnunetv2.utilities.plans_handling.plans_handler import ConfigurationManager, PlansManager
from dynamic_network_architectures.building_blocks.helper import get_matching_instancenorm, convert_dim_to_conv_op
from dynamic_network_architectures.initialization.weight_init import init_last_bn_before_add_to_0
from nnunetv2.utilities.network_initialization import InitWeights_He
from mamba_ssm import Mamba
import copy
import warnings

class BiPixelMambaLayer(nn.Module):
    def __init__(self, dim, p, d_state = 16, d_conv = 4, expand = 2):
        super().__init__()
        self.dim = dim
        self.norm = nn.LayerNorm(dim)


        self.mamba_forw = Mamba(
                d_model=dim, # Model dimension d_model
                d_state=d_state,  # SSM state expansion factor
                d_conv=d_conv,    # Local convolution width
                expand=expand,    # Block expansion factor
                use_fast_path=False,
        )

     
        self.out_proj = copy.deepcopy(self.mamba_forw.out_proj)


        
        self.mamba_backw = Mamba(
                d_model=dim, # Model dimension d_model
                d_state=d_state,  # SSM state expansion factor
                d_conv=d_conv,    # Local convolution width
                expand=expand,    # Block expansion factor
                use_fast_path=False,

        )

      
        # adjust the window size here to fit the feature map
        self.p = p*5
        self.p1 = 5*p
        self.p2 = 7*p
        self.p3 = 6*p
        self.mamba_forw.out_proj = nn.Identity()
        self.mamba_backw.out_proj = nn.Identity()
       


    @autocast(enabled=False)
    def forward(self, x):
        if x.dtype == torch.float16:
            x = x.type(torch.float32)

        ll = len(x.shape)

        B, C = x.shape[:2]

        assert C == self.dim
        img_dims = x.shape[2:]

        if ll == 5: #3d
         
            Z,H,W = x.shape[2:]

            if Z%self.p1==0 and H%self.p2==0 and W%self.p3==0:
                x_div = x.reshape(B, C, Z//self.p1, self.p1, H//self.p2, self.p2, W//self.p3, self.p3)
                x_div = x_div.permute(0, 3, 5, 7, 1, 2, 4, 6).contiguous().view(B*self.p1*self.p2*self.p3, C, Z//self.p1, H//self.p2, W//self.p3)
            else:
                x_div = x

        elif ll == 4: #2d
            H,W = x.shape[2:]

            if H%self.p==0 and W%self.p==0:                
                x_div = x.reshape(B, C, H//self.p, self.p, W//self.p, self.p).permute(0, 3, 5, 1, 2, 4).contiguous().view(B*self.p*self.p, C, H//self.p, W//self.p)            
            else:
                x_div = x
        

        NB = x_div.shape[0]
        if ll == 5: #3d
            NZ,NH,NW = x_div.shape[2:]
        else:
            NH,NW = x_div.shape[2:]

        n_tokens = x_div.shape[2:].numel()
   

        x_flat = x_div.reshape(NB, C, n_tokens).transpose(-1, -2)
        x_norm = self.norm(x_flat)

        y_norm = torch.flip(x_norm, dims=[1])

        x_mamba = self.mamba_forw(x_norm)
        y_mamba = self.mamba_backw(y_norm)

        x_out = self.out_proj(x_mamba + torch.flip(y_mamba, dims=[1]))

        if ll == 5:
            if Z%self.p1==0 and H%self.p2==0 and W%self.p3==0:
                x_out = x_out.transpose(-1, -2).reshape(B, self.p1, self.p2, self.p3, C, NZ, NH, NW).permute(0, 4, 5, 1, 6, 2, 7, 3).contiguous().reshape(B, C, *img_dims)
            else:
                x_out = x_out.transpose(-1, -2).reshape(B, C, *img_dims)
        if ll == 4:
            if H%self.p==0 and W%self.p==0:
                x_out = x_out.transpose(-1, -2).reshape(B, self.p, self.p, C, NH, NW).permute(0, 3, 4, 1, 5, 2).contiguous().reshape(B, C, *img_dims)
            else:
                x_out = x_out.transpose(-1, -2).reshape(B, C, *img_dims)
        out = x_out + x

        return out

class BiWindowMambaLayer(nn.Module):
    def __init__(self, dim, p, d_state = 16, d_conv = 4, expand = 2):
        super().__init__()
        self.dim = dim
        self.norm = nn.LayerNorm(dim)

        self.p = p
        self.mamba_forw = Mamba(
                d_model=dim, # Model dimension d_model
                d_state=d_state,  # SSM state expansion factor
                d_conv=d_conv,    # Local convolution width
                expand=expand,    # Block expansion factor
                use_fast_path=False,
        )

     
        self.out_proj = copy.deepcopy(self.mamba_forw.out_proj)

        

        
        self.mamba_backw = Mamba(
                d_model=dim, # Model dimension d_model
                d_state=d_state,  # SSM state expansion factor
                d_conv=d_conv,    # Local convolution width
                expand=expand,    # Block expansion factor
                use_fast_path=False,

        )

      

       
        self.mamba_forw.out_proj = nn.Identity()
        self.mamba_backw.out_proj = nn.Identity()
       


    @autocast(enabled=False)
    def forward(self, x):
        if x.dtype == torch.float16:
            x = x.type(torch.float32)

        ll = len(x.shape)
     
       

        B, C = x.shape[:2]

        assert C == self.dim
   
        img_dims = x.shape[2:]



        if ll == 5: #3d
            
            Z,H,W = x.shape[2:]

            if Z%self.p==0 and H%self.p==0 and W%self.p==0:
                pool_layer = nn.AvgPool3d(self.p, stride=self.p)
                x_div = pool_layer(x)
            else:
                x_div = x

        elif ll == 4: #2d

            H,W = x.shape[2:]
            if self.p==0:
                self.p = 1
            if H%self.p==0 and W%self.p==0:
                pool_layer = nn.AvgPool2d(self.p, stride=self.p)
                x_div = pool_layer(x)
            else:
                x_div = x
        

      
        if ll == 5: #3d
            NZ,NH,NW = x_div.shape[2:]
        else:
            NH,NW = x_div.shape[2:]

        n_tokens = x_div.shape[2:].numel()
   

        x_flat = x_div.reshape(B, C, n_tokens).transpose(-1, -2)
        x_norm = self.norm(x_flat)

        y_norm = torch.flip(x_norm, dims=[1])

        x_mamba = self.mamba_forw(x_norm)
        y_mamba = self.mamba_backw(y_norm)

        x_out = self.out_proj(x_mamba + torch.flip(y_mamba, dims=[1]))

        if ll == 5:
            if Z%self.p==0 and H%self.p==0 and W%self.p==0:
                unpool_layer = nn.Upsample(scale_factor=self.p, mode='nearest')
                x_out = x_out.transpose(-1, -2).reshape(B, C, NZ, NH, NW)
                x_out = unpool_layer(x_out)
            else:
                x_out = x_out.transpose(-1, -2).reshape(B, C, *img_dims)
        if ll == 4:
            if self.p==0:
                self.p = 1
            if H%self.p==0 and W%self.p==0:
                unpool_layer = nn.Upsample(scale_factor=self.p, mode='nearest')
                x_out = x_out.transpose(-1, -2).reshape(B, C, NH, NW)
                x_out = unpool_layer(x_out)
            else:
                x_out = x_out.transpose(-1, -2).reshape(B, C, *img_dims)
                
        out = x_out + x

        return out

class ResidualBiMambaEncoder(nn.Module):
    def __init__(self,
                 input_channels: int,
                 n_stages: int,
                 features_per_stage: Union[int, List[int], Tuple[int, ...]],
                 conv_op: Type[_ConvNd],
                 kernel_sizes: Union[int, List[int], Tuple[int, ...]],
                 strides: Union[int, List[int], Tuple[int, ...], Tuple[Tuple[int, ...], ...]],
                 n_blocks_per_stage: Union[int, List[int], Tuple[int, ...]],
                 conv_bias: bool = False,
                 norm_op: Union[None, Type[nn.Module]] = None,
                 norm_op_kwargs: dict = None,
                 dropout_op: Union[None, Type[_DropoutNd]] = None,
                 dropout_op_kwargs: dict = None,
                 nonlin: Union[None, Type[torch.nn.Module]] = None,
                 nonlin_kwargs: dict = None,
                 block: Union[Type[BasicBlockD], Type[BottleneckD]] = BasicBlockD,
                 bottleneck_channels: Union[int, List[int], Tuple[int, ...]] = None,
                 return_skips: bool = False,
                 disable_default_stem: bool = False,
                 stem_channels: int = None,
                 pool_type: str = 'conv',
                 stochastic_depth_p: float = 0.0,
                 squeeze_excitation: bool = False,
                 squeeze_excitation_reduction_ratio: float = 1. / 16
                 ):
        super().__init__()
        if isinstance(kernel_sizes, int):
            kernel_sizes = [kernel_sizes] * n_stages
        if isinstance(features_per_stage, int):
            features_per_stage = [features_per_stage] * n_stages
        if isinstance(n_blocks_per_stage, int):
            n_blocks_per_stage = [n_blocks_per_stage] * n_stages
        if isinstance(strides, int):
            strides = [strides] * n_stages
        if bottleneck_channels is None or isinstance(bottleneck_channels, int):
            bottleneck_channels = [bottleneck_channels] * n_stages
        assert len(
            bottleneck_channels) == n_stages, "bottleneck_channels must be None or have as many entries as we have resolution stages (n_stages)"
        assert len(
            kernel_sizes) == n_stages, "kernel_sizes must have as many entries as we have resolution stages (n_stages)"
        assert len(
            n_blocks_per_stage) == n_stages, "n_conv_per_stage must have as many entries as we have resolution stages (n_stages)"
        assert len(
            features_per_stage) == n_stages, "features_per_stage must have as many entries as we have resolution stages (n_stages)"
        assert len(strides) == n_stages, "strides must have as many entries as we have resolution stages (n_stages). " \
                                         "Important: first entry is recommended to be 1, else we run strided conv drectly on the input"

        pool_op = get_matching_pool_op(conv_op, pool_type=pool_type) if pool_type != 'conv' else None

        # build a stem, Todo maybe we need more flexibility for this in the future. For now, if you need a custom
        #  stem you can just disable the stem and build your own.
        #  THE STEM DOES NOT DO STRIDE/POOLING IN THIS IMPLEMENTATION
        if not disable_default_stem:
            if stem_channels is None:
                stem_channels = features_per_stage[0]
            self.stem = StackedConvBlocks(1, conv_op, input_channels, stem_channels, kernel_sizes[0], 1, conv_bias,
                                          norm_op, norm_op_kwargs, dropout_op, dropout_op_kwargs, nonlin, nonlin_kwargs)
            input_channels = stem_channels
        else:
            self.stem = None

        # now build the network
        stages = []
        w_mamba_layers = []
        mamba_layers = []
        for s in range(n_stages):
            stride_for_conv = strides[s] if pool_op is None else 1

            stage = StackedResidualBlocks(
                n_blocks_per_stage[s], conv_op, input_channels, features_per_stage[s], kernel_sizes[s], stride_for_conv,
                conv_bias, norm_op, norm_op_kwargs, dropout_op, dropout_op_kwargs, nonlin, nonlin_kwargs,
                block=block, bottleneck_channels=bottleneck_channels[s], stochastic_depth_p=stochastic_depth_p,
                squeeze_excitation=squeeze_excitation,
                squeeze_excitation_reduction_ratio=squeeze_excitation_reduction_ratio
            )

            if pool_op is not None:
                stage = nn.Sequential(pool_op(strides[s]), stage)

            stages.append(stage)
            input_channels = features_per_stage[s]
            
            mamba_layers.append(BiPixelMambaLayer(input_channels, 2**( (n_stages-s+1)//2-1) ))
            w_mamba_layers.append(BiWindowMambaLayer(input_channels, 2**((n_stages-s+1)//2)//2 ))
            


        #self.stages = nn.Sequential(*stages)
        self.stages = nn.ModuleList(stages)
        self.output_channels = features_per_stage
        self.strides = [maybe_convert_scalar_to_list(conv_op, i) for i in strides]
        self.return_skips = return_skips

        # we store some things that a potential decoder needs
        self.conv_op = conv_op
        self.norm_op = norm_op
        self.norm_op_kwargs = norm_op_kwargs
        self.nonlin = nonlin
        self.nonlin_kwargs = nonlin_kwargs
        self.dropout_op = dropout_op
        self.dropout_op_kwargs = dropout_op_kwargs
        self.conv_bias = conv_bias
        self.kernel_sizes = kernel_sizes

        self.mamba_layers = nn.ModuleList(mamba_layers)
        self.w_mamba_layers = nn.ModuleList(w_mamba_layers)

    def forward(self, x):
        if self.stem is not None:
            x = self.stem(x)
        ret = []
        #for s in self.stages:
        for s in range(len(self.stages)):
            #x = s(x)
            x = self.stages[s](x)
            x = self.mamba_layers[s](x)
            x = self.w_mamba_layers[s](x)
            ret.append(x)
        if self.return_skips:
            return ret
        else:
            return ret[-1]

    def compute_conv_feature_map_size(self, input_size):
        if self.stem is not None:
            output = self.stem.compute_conv_feature_map_size(input_size)
        else:
            output = np.int64(0)

        for s in range(len(self.stages)):
            output += self.stages[s].compute_conv_feature_map_size(input_size)
            input_size = [i // j for i, j in zip(input_size, self.strides[s])]

        return output

class UNetResDecoder(nn.Module):
    def __init__(self,
                 encoder: Union[PlainConvEncoder, ResidualBiMambaEncoder],
                 num_classes: int,
                 n_conv_per_stage: Union[int, Tuple[int, ...], List[int]],
                 deep_supervision, nonlin_first: bool = False):
        """
        This class needs the skips of the encoder as input in its forward.

        the encoder goes all the way to the bottleneck, so that's where the decoder picks up. stages in the decoder
        are sorted by order of computation, so the first stage has the lowest resolution and takes the bottleneck
        features and the lowest skip as inputs
        the decoder has two (three) parts in each stage:
        1) conv transpose to upsample the feature maps of the stage below it (or the bottleneck in case of the first stage)
        2) n_conv_per_stage conv blocks to let the two inputs get to know each other and merge
        3) (optional if deep_supervision=True) a segmentation output Todo: enable upsample logits?
        :param encoder:
        :param num_classes:
        :param n_conv_per_stage:
        :param deep_supervision:
        """
        super().__init__()
        self.deep_supervision = deep_supervision
        self.encoder = encoder
        self.num_classes = num_classes
        n_stages_encoder = len(encoder.output_channels)
        if isinstance(n_conv_per_stage, int):
            n_conv_per_stage = [n_conv_per_stage] * (n_stages_encoder - 1)
        assert len(n_conv_per_stage) == n_stages_encoder - 1, "n_conv_per_stage must have as many entries as we have " \
                                                          "resolution stages - 1 (n_stages in encoder - 1), " \
                                                          "here: %d" % n_stages_encoder

        transpconv_op = get_matching_convtransp(conv_op=encoder.conv_op)

        # we start with the bottleneck and work out way up
        stages = []
        transpconvs = []
        seg_layers = []
        for s in range(1, n_stages_encoder):
            input_features_below = encoder.output_channels[-s]
            input_features_skip = encoder.output_channels[-(s + 1)]
            stride_for_transpconv = encoder.strides[-s]
            transpconvs.append(transpconv_op(
                input_features_below, input_features_skip, stride_for_transpconv, stride_for_transpconv,
                bias=encoder.conv_bias
            ))
            # input features to conv is 2x input_features_skip (concat input_features_skip with transpconv output)
            stages.append(StackedResidualBlocks(
                n_blocks = n_conv_per_stage[s-1],
                conv_op = encoder.conv_op,
                input_channels = 2 * input_features_skip,
                output_channels = input_features_skip,
                kernel_size = encoder.kernel_sizes[-(s + 1)],
                initial_stride = 1,
                conv_bias = encoder.conv_bias,
                norm_op = encoder.norm_op,
                norm_op_kwargs = encoder.norm_op_kwargs,
                dropout_op = encoder.dropout_op,
                dropout_op_kwargs = encoder.dropout_op_kwargs,
                nonlin = encoder.nonlin,
                nonlin_kwargs = encoder.nonlin_kwargs,
            ))
            # we always build the deep supervision outputs so that we can always load parameters. If we don't do this
            # then a model trained with deep_supervision=True could not easily be loaded at inference time where
            # deep supervision is not needed. It's just a convenience thing
            seg_layers.append(encoder.conv_op(input_features_skip, num_classes, 1, 1, 0, bias=True))

        self.stages = nn.ModuleList(stages)
        self.transpconvs = nn.ModuleList(transpconvs)
        self.seg_layers = nn.ModuleList(seg_layers)

    def forward(self, skips):
        """
        we expect to get the skips in the order they were computed, so the bottleneck should be the last entry
        :param skips:
        :return:
        """
        lres_input = skips[-1]
        seg_outputs = []
        for s in range(len(self.stages)):
            x = self.transpconvs[s](lres_input)
            x = torch.cat((x, skips[-(s+2)]), 1)
            x = self.stages[s](x)
            if self.deep_supervision:
                seg_outputs.append(self.seg_layers[s](x))
            elif s == (len(self.stages) - 1):
                seg_outputs.append(self.seg_layers[-1](x))
            lres_input = x

        # invert seg outputs so that the largest segmentation prediction is returned first
        seg_outputs = seg_outputs[::-1]

        if not self.deep_supervision:
            r = seg_outputs[0]
        else:
            r = seg_outputs
        return r

class LKMUNet(nn.Module):
    def __init__(self,
                 input_channels: int,
                 n_stages: int,
                 features_per_stage: Union[int, List[int], Tuple[int, ...]],
                 conv_op: Type[_ConvNd],
                 kernel_sizes: Union[int, List[int], Tuple[int, ...]],
                 strides: Union[int, List[int], Tuple[int, ...]],
                 n_conv_per_stage: Union[int, List[int], Tuple[int, ...]],
                 num_classes: int,
                 n_conv_per_stage_decoder: Union[int, Tuple[int, ...], List[int]],
                 conv_bias: bool = False,
                 norm_op: Union[None, Type[nn.Module]] = None,
                 norm_op_kwargs: dict = None,
                 dropout_op: Union[None, Type[_DropoutNd]] = None,
                 dropout_op_kwargs: dict = None,
                 nonlin: Union[None, Type[torch.nn.Module]] = None,
                 nonlin_kwargs: dict = None,
                 deep_supervision: bool = False,
                 block: Union[Type[BasicBlockD], Type[BottleneckD]] = BasicBlockD,
                 bottleneck_channels: Union[int, List[int], Tuple[int, ...]] = None,
                 stem_channels: int = None
                 ):
        super().__init__()
        n_blocks_per_stage = n_conv_per_stage
        if isinstance(n_blocks_per_stage, int):
            n_blocks_per_stage = [n_blocks_per_stage] * n_stages
        if isinstance(n_conv_per_stage_decoder, int):
            n_conv_per_stage_decoder = [n_conv_per_stage_decoder] * (n_stages - 1)
        assert len(n_blocks_per_stage) == n_stages, "n_blocks_per_stage must have as many entries as we have " \
                                                  f"resolution stages. here: {n_stages}. " \
                                                  f"n_blocks_per_stage: {n_blocks_per_stage}"
        assert len(n_conv_per_stage_decoder) == (n_stages - 1), "n_conv_per_stage_decoder must have one less entries " \
                                                                f"as we have resolution stages. here: {n_stages} " \
                                                                f"stages, so it should have {n_stages - 1} entries. " \
                                                                f"n_conv_per_stage_decoder: {n_conv_per_stage_decoder}"
        self.encoder = ResidualBiMambaEncoder(input_channels, n_stages, features_per_stage, conv_op, kernel_sizes, strides,
                                       n_blocks_per_stage, conv_bias, norm_op, norm_op_kwargs, dropout_op,
                                       dropout_op_kwargs, nonlin, nonlin_kwargs, block, bottleneck_channels,
                                       return_skips=True, disable_default_stem=False, stem_channels=stem_channels)
        self.decoder = UNetResDecoder(self.encoder, num_classes, n_conv_per_stage_decoder, deep_supervision)

    def forward(self, x):
        skips = self.encoder(x)
        out   = self.decoder(skips)          # tensor 或 list

        # ─── 限制到 [0, 1] ───
        if isinstance(out, (list, tuple)):   # 若有 deep supervision
            out = [torch.sigmoid(o) for o in out]
        else:
            out = torch.sigmoid(out)
        return out

    def compute_conv_feature_map_size(self, input_size):
        assert len(input_size) == convert_conv_op_to_dim(self.encoder.conv_op), "just give the image size without color/feature channels or " \
                                                                                "batch channel. Do not give input_size=(b, c, x, y(, z)). " \
                                                                                "Give input_size=(x, y(, z))!"
        return self.encoder.compute_conv_feature_map_size(input_size) + self.decoder.compute_conv_feature_map_size(input_size)
    
# ---------- 1. 建立 LKMUNet ----------
cfg = dict(
    input_channels=1,
    n_stages=4,
    features_per_stage=[32, 64, 128, 256],
    conv_op=nn.Conv2d,
    kernel_sizes=[3, 3, 3, 3],
    strides=[1, 2, 2, 2],
    n_conv_per_stage=[2, 2, 2, 2],
    num_classes=1,
    n_conv_per_stage_decoder=[2, 2, 2],
    deep_supervision=False,
    conv_bias=False,
    norm_op=nn.BatchNorm2d,
    norm_op_kwargs=dict(eps=1e-5, affine=True),
    dropout_op=None,
    dropout_op_kwargs=None,
    nonlin=nn.LeakyReLU,
    nonlin_kwargs=dict(inplace=False),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("⛔ 目前裝置沒有可用的 CUDA，無法執行含 Mamba 的 LKMUNet。")

model = LKMUNet(**cfg).eval().to(device)

# ---------- 2. 隨機輸入 ----------
x = torch.randn(2, 1, 128, 128, device=device)

# ---------- 3. 前向傳遞 & 範圍檢查 ----------
with torch.no_grad():
    y = model(x)

print(f"output range: {y.min().item():.4f} – {y.max().item():.4f}")
assert (0.0 <= y.min() <= 1.0) and (0.0 <= y.max() <= 1.0), \
       "Sigmoid 限制失效！"

print("✅ Sigmoid test passed on CUDA.")

In [ ]:
# LKMUNet-ODE-FiLM (DecoderSigmoid / Drift field Regular / FiLM Constraint)
# ================================================================
#  • Decoder 加 Sigmoid     👉 Output ∈ [0, 1]
#  • Drift μ(x) clamp ±3    👉 避免 ODE 爆炸
#  • FiLM γ, β → tanh 壓縮；β×0.1
#  • 其餘結構與舊版相容（仍用 Euler 固定步；若要 adaptive 可再調）
# ================================================================
import torch, torch.nn as nn, torch.nn.functional as F
from torchdiffeq import odeint, odeint_adjoint          # pip install torchdiffeq
from typing import Tuple

# ────────────────── 小工具 ───────────────────────────────────────
def patch_inplace_relu(mod: nn.Module):
    for n, c in mod.named_children():
        if isinstance(c, (nn.ReLU, nn.LeakyReLU)) and getattr(c, "inplace", False):
            setattr(mod, n, c.__class__(inplace=False))
        else:
            patch_inplace_relu(c)

# ────────────────── ODE drift (FiLM) ────────────────────────────
class SpatialODEFunc(nn.Module):
    def __init__(self, C: int):
        super().__init__(); self.C = C
        self.conv = nn.Sequential(
            nn.Conv2d(C, C, 3, 1, 1, bias=False),
            nn.ReLU(inplace=False),
            nn.Conv2d(C, C, 3, 1, 1, bias=False)
        )
        self.gamma = self.beta = None

    def set_film(self, γ, β):
        self.gamma, self.beta = γ, β

    def forward(self, t, x_flat):                     # dX/dt
        B, D = x_flat.shape; H = W = int((D // self.C) ** .5)
        x = x_flat.view(B, self.C, H, W)
        out = self.conv(x)
        if self.gamma is not None:
            out = self.gamma * out + self.beta
        ## --- FIX ②：clamp drift --------------------------------------------------
        out = torch.clamp(out, -3., 3.)
        return out.flatten(1)

# ────────────────── ODEBlock ────────────────────────────────────
class ODEBlock(nn.Module):
    def __init__(self, func, *, steps=8,
                 rtol=1e-3, atol=1e-3, method="euler", adjoint=True):
        super().__init__(); self.func, self.N, self.rtol, self.atol,\
            self.method, self.adjoint = func, steps, rtol, atol, method, adjoint
    def forward(self, x_map, int_t, γ=None, β=None):
        self.func.set_film(γ, β)
        B, C, H, W = x_map.shape
        x0 = x_map.flatten(1)
        odeint_fn = odeint_adjoint if self.adjoint else odeint
        out = odeint_fn(
            self.func, x0, int_t,
            method=self.method, rtol=self.rtol, atol=self.atol,
            options=dict(step_size=1. / self.N)
        )
        return out[-1].view(B, C, H, W)

# ────────────────── 主模型 (換 ODEBlock) ───────────────────────
class LKMUNetODE_FiLM(nn.Module):
    def __init__(self, backbone, mid_encoder, sde_dim,
                 *, reduce=True, steps=8, strategy="auto"):
        super().__init__()
        self.backbone, self.mid_encoder, self.strategy = backbone, mid_encoder, strategy
        patch_inplace_relu(backbone)
        if mid_encoder is not None:
            patch_inplace_relu(mid_encoder)

        C_backbone = backbone.encoder.output_channels[-1]
        self.reduce_conv = nn.Identity() if (not reduce or C_backbone == sde_dim) \
                           else nn.Conv2d(C_backbone, sde_dim, 1, bias=False)
        self.down_pool = nn.AvgPool2d(2)
        self.ode_block = ODEBlock(
            SpatialODEFunc(sde_dim),
            steps=steps, rtol=1e-3, atol=1e-3,
            method="euler", adjoint=True
        )

        if mid_encoder is not None:
            self.mid_reduce  = nn.Identity() if C_backbone == sde_dim \
                               else nn.Conv2d(C_backbone, sde_dim, 1, bias=False)
            self.film_gamma  = nn.Conv2d(sde_dim, sde_dim, 1)
            self.film_beta   = nn.Conv2d(sde_dim, sde_dim, 1)
            self.concat_reduce = None           # lazy build

        self.up_bridge = nn.ConvTranspose2d(sde_dim, C_backbone, 2, 2, 0, bias=False)

    # -------- gamma / beta ----------------------------------------------
    def _prepare_film(self, mid_pet: torch.Tensor, hw: Tuple[int, int]):
        if self.mid_encoder is None:
            return None, None
        if mid_pet.ndim == 5:
            B, M, C0, H, W = mid_pet.shape
        else:
            B, C0, H, W = mid_pet.shape; M = 1

        strat = self.strategy
        if strat == "auto":
            enc_in = next(m.in_channels for m in self.mid_encoder.modules()
                          if isinstance(m, nn.Conv2d))
            strat = "concat" if (M * C0 == enc_in) else "stack"

        # concat -----------------------------------------------------------
        if strat == "concat":
            if mid_pet.ndim == 5:
                mid_pet = mid_pet.view(B, M * C0, H, W)
            enc_in = next(m.in_channels for m in self.mid_encoder.modules()
                          if isinstance(m, nn.Conv2d))
            if mid_pet.size(1) != enc_in:
                if (self.concat_reduce is None) or (self.concat_reduce.in_channels != mid_pet.size(1)):
                    self.concat_reduce = nn.Conv2d(mid_pet.size(1), enc_in, 1).to(mid_pet.device)
                mid_pet = self.concat_reduce(mid_pet)

        # stack ------------------------------------------------------------
        elif strat == "stack" and mid_pet.ndim == 5:
            mid_pet = mid_pet.view(B * M, C0, H, W)

        # encode mid frames -----------------------------------------------
        mid_feat = self.mid_encoder.encoder(mid_pet)[-1]
        mid_feat = F.interpolate(mid_feat, size=hw, mode="bilinear", align_corners=False)
        mid_feat = self.mid_reduce(mid_feat)

        if strat == "stack" and M > 1:
            C_lat = mid_feat.size(1)
            mid_feat = mid_feat.view(B, M, C_lat, *hw).mean(1)

        γ = torch.tanh(self.film_gamma(mid_feat))          # --- FIX ③
        β = 0.1 * torch.tanh(self.film_beta(mid_feat))     # --- FIX ③
        return γ, β

    # -------- forward ----------------------------------------------------
    def forward(self, early_pet, t, *, mid_pet=None):
        skips = self.backbone.encoder(early_pet)
        x_bot = skips[-1]
        x_low = self.down_pool(self.reduce_conv(x_bot))     # [B,sde_dim,16,16]
        Hp, Wp = x_low.shape[2:]

        γ = β = None
        if (mid_pet is not None) and (self.mid_encoder is not None):
            γ, β = self._prepare_film(mid_pet, (Hp, Wp))

        t_val = float(t.item()) if isinstance(t, torch.Tensor) else float(t)
        int_t = torch.tensor([0., t_val], device=early_pet.device, dtype=early_pet.dtype)
        x_low_out = self.ode_block(x_low, int_t, γ, β)
        skips[-1] = self.up_bridge(x_low_out).contiguous()

        pred = self.backbone.decoder(skips)
        pred = torch.sigmoid(pred)                         # --- FIX ①
        return pred, x_low_out, γ, β

# ────────────────── Builder (mid-encoder 同架構) ────────────────
def build_lkmunet_ode_film(cfg: dict, *, strategy="auto"):
    net = LKMUNet(**cfg)
    mid = LKMUNet(**cfg)
    return LKMUNetODE_FiLM(
        net, mid,
        sde_dim=cfg['features_per_stage'][-1] // 2,
        reduce=True, steps=8, strategy=strategy
    )

# ────────────────── quick smoke test ───────────────────────────
if __name__ == "__main__":
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    cfg = dict(
        input_channels=1, n_stages=4, features_per_stage=[32, 64, 128, 256],
        conv_op=nn.Conv2d, kernel_sizes=[(3,3)]*4,
        strides=[(1,1),(2,2),(2,2),(2,2)],
        n_conv_per_stage=[2]*4, num_classes=1,
        n_conv_per_stage_decoder=[2,2,2],
        conv_bias=False, norm_op=nn.BatchNorm2d,
        norm_op_kwargs=dict(eps=1e-5, affine=True),
        dropout_op=None, dropout_op_kwargs=None,
        nonlin=nn.LeakyReLU, nonlin_kwargs=dict(inplace=False),
        deep_supervision=False
    )

    model = build_lkmunet_ode_film(cfg, strategy="stack").to(device).eval()
    B, M = 2, 3
    x_early = torch.randn(B, 1, 128, 128, device=device)
    x_mid   = torch.randn(B, M, 1, 128, 128, device=device)
    with torch.no_grad():
        y, *_ = model(x_early, t=1.0, mid_pet=x_mid)
    print("Output range:", y.min().item(), y.max().item(), "| shape:", y.shape)


In [ ]:
# LKMUNet-SDE-FiLM (DecoderSigmoid / Drift field Regular / FiLM Constraint)
# ================================================================
#   1) Decoder 輸出加 Sigmoid（Output ∈ [0,1]）
#   2) Drift field μ(x) 以 clamp(−3,3) 正規化，並在 loss 提高 smoothness 權重
#   3) FiLM γ, β → tanh 壓縮；β 再乘 0.1
#   4) 其餘框架保持不變（仍用 Euler 8 steps，方便與舊 ckpt 相容）
# ================================================================
import torch, torch.nn as nn, torch.nn.functional as F, torchsde
from typing import Tuple

# ---------- 小工具 ------------------------------------------------
def patch_inplace_relu(mod: nn.Module):
    for n, c in mod.named_children():
        if isinstance(c, (nn.ReLU, nn.LeakyReLU)) and getattr(c, "inplace", False):
            setattr(mod, n, c.__class__(inplace=False))
        else:
            patch_inplace_relu(c)

# ---------- SDE ---------------------------------------------------
class SpatialSDEFunc(nn.Module):
    noise_type, sde_type = "diagonal", "ito"
    def __init__(self, C):
        super().__init__(); self.C = C
        self.mu = nn.Sequential(
            nn.Conv2d(C, C, 3, 1, 1, bias=False),
            nn.ReLU(inplace=False),
            nn.Conv2d(C, C, 3, 1, 1, bias=False)
        )
        self.sigma = nn.Conv2d(C, C, 1, bias=False)
        self.gamma = self.beta = None
    def set_film(self, g, b): self.gamma, self.beta = g, b

    def f(self, t, x):
        B, D = x.shape; H = W = int((D // self.C) ** .5)
        feat = x.view(B, self.C, H, W)
        out  = self.mu(feat)
        if self.gamma is not None:
            out = self.gamma * out + self.beta
        ## --- FIX ②：clamp drift ------------------------------------------------
        out  = torch.clamp(out, min=-3., max=3.)
        return out.flatten(1)

    def g(self, t, x):
        B, D = x.shape; H = W = int((D // self.C) ** .5)
        return self.sigma(x.view(B, self.C, H, W)).flatten(1)

class SDEBlock(nn.Module):
    def __init__(self, func, *, steps=8, tol=1e-2, adjoint=True):
        super().__init__(); self.func, self.steps, self.tol, self.adjoint = func, steps, tol, adjoint
    def forward(self, x_map, int_t, g=None, b=None):
        self.func.set_film(g, b)
        B, C, H, W = x_map.shape
        x_vec = x_map.flatten(1)
        solver = torchsde.sdeint_adjoint if self.adjoint else torchsde.sdeint
        out = solver(self.func, x_vec, int_t,
                     dt=1./self.steps, method="euler",
                     rtol=self.tol, atol=self.tol)
        return out[-1].view(B, C, H, W)

# ---------- 主模型 ------------------------------------------------
class LKMUNetSDE_FiLM(nn.Module):
    def __init__(self, backbone, mid_encoder, sde_dim,
                 *, reduce=True, steps=8, strategy="auto"):
        super().__init__()
        self.backbone, self.mid_encoder, self.strategy = backbone, mid_encoder, strategy
        patch_inplace_relu(backbone)
        if mid_encoder is not None: patch_inplace_relu(mid_encoder)

        C_backbone = backbone.encoder.output_channels[-1]
        self.reduce_conv = nn.Identity() if (not reduce or C_backbone == sde_dim) \
                           else nn.Conv2d(C_backbone, sde_dim, 1, bias=False)
        self.down_pool  = nn.AvgPool2d(2)
        self.sde_block  = SDEBlock(SpatialSDEFunc(sde_dim), steps=steps, tol=1e-2, adjoint=True)

        if mid_encoder is not None:
            self.mid_reduce  = nn.Identity() if C_backbone == sde_dim \
                               else nn.Conv2d(C_backbone, sde_dim, 1, bias=False)
            self.film_gamma  = nn.Conv2d(sde_dim, sde_dim, 1)
            self.film_beta   = nn.Conv2d(sde_dim, sde_dim, 1)
            self.concat_reduce = None           # lazy build

        self.up_bridge = nn.ConvTranspose2d(sde_dim, C_backbone, 2, 2, 0, bias=False)

    # --------- 產生 γ / β ----------------------------------------
    def _prepare_film(self, mid_pet: torch.Tensor, hw: Tuple[int, int]):
        if self.mid_encoder is None: return None, None
        if mid_pet.ndim == 5:                       # [B,M,C,H,W]
            B, M, C0, H, W = mid_pet.shape
        else:
            B, C0, H, W = mid_pet.shape; M = 1

        strat = self.strategy
        if strat == "auto":
            enc_in = next(m.in_channels for m in self.mid_encoder.modules()
                          if isinstance(m, nn.Conv2d))
            strat = "concat" if (M * C0 == enc_in) else "stack"

        # concat ---------------------------------------------------
        if strat == "concat":
            if mid_pet.ndim == 5:
                mid_pet = mid_pet.view(B, M * C0, H, W)
            enc_in = next(m.in_channels for m in self.mid_encoder.modules()
                          if isinstance(m, nn.Conv2d))
            if mid_pet.size(1) != enc_in:
                if (self.concat_reduce is None) or (self.concat_reduce.in_channels != mid_pet.size(1)):
                    self.concat_reduce = nn.Conv2d(mid_pet.size(1), enc_in, 1).to(mid_pet.device)
                mid_pet = self.concat_reduce(mid_pet)

        # stack ----------------------------------------------------
        elif strat == "stack" and mid_pet.ndim == 5:
            mid_pet = mid_pet.view(B * M, C0, H, W)

        # encode mid frames ---------------------------------------
        mid_feat = self.mid_encoder.encoder(mid_pet)[-1]
        mid_feat = F.interpolate(mid_feat, size=hw, mode="bilinear", align_corners=False)
        mid_feat = self.mid_reduce(mid_feat)       # → sde_dim

        if strat == "stack" and M > 1:
            C_lat = mid_feat.size(1)
            mid_feat = mid_feat.view(B, M, C_lat, *hw).mean(1)

        γ = torch.tanh(self.film_gamma(mid_feat))           # --- FIX ③
        β = 0.1 * torch.tanh(self.film_beta(mid_feat))      # --- FIX ③
        return γ, β

    # --------- forward ------------------------------------------
    def forward(self, early_pet, t, *, mid_pet=None):
        skips = self.backbone.encoder(early_pet)
        x_bot = skips[-1]                                   # [B,C,32,32]
        x_low = self.down_pool(self.reduce_conv(x_bot))     # [B,sde_dim,16,16]
        Hp, Wp = x_low.shape[2:]

        γ = β = None
        if mid_pet is not None and self.mid_encoder is not None:
            γ, β = self._prepare_film(mid_pet, (Hp, Wp))

        t_val  = float(t.item()) if isinstance(t, torch.Tensor) else float(t)
        int_t  = torch.tensor([0., t_val], device=early_pet.device, dtype=early_pet.dtype)
        x_low_out = self.sde_block(x_low, int_t, γ, β)
        skips[-1] = self.up_bridge(x_low_out).contiguous()  # 回到 32×32

        pred = self.backbone.decoder(skips)
        pred = torch.sigmoid(pred)                          # --- FIX ①
        return pred, x_low_out, γ, β

# ---------- Builder ------------------------------------------------------
def build_lkmunet_sde_film(cfg: dict, *, strategy="auto"):
    net = LKMUNet(**cfg)
    mid = LKMUNet(**cfg)           # 可選擇凍結
    return LKMUNetSDE_FiLM(net, mid,
                           sde_dim=cfg['features_per_stage'][-1] // 2,
                           reduce=True, steps=8, strategy=strategy)

# ---------- loss（smoothness 權重 ↑） ------------------------------------
class ImageFlowNetLoss(nn.Module):
    def __init__(self, smooth_w=5e-4, cont_w=1.):           # --- FIX ②
        super().__init__()
        self.mse  = nn.MSELoss()
        self.w_sm = smooth_w
        self.w_c  = cont_w
    def forward(self, pred, tgt, drift=None, early=None, mid=None):
        loss = self.mse(pred, tgt)
        if drift is not None:
            loss += self.w_sm * drift.pow(2).mean()
        if (early is not None) and (mid is not None):
            e = F.normalize(early, p=2, dim=1)
            m = F.normalize(mid,   p=2, dim=1)
            loss += self.w_c * (1 - (e*m).sum(1).mean())
        return loss

# ---------- quick test ---------------------------------------------------
if __name__ == "__main__":
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    cfg = dict(
        input_channels=1, n_stages=4, #features_per_stage=[64,128,256,512],
        features_per_stage= [32, 64, 128, 256],
        conv_op=nn.Conv2d, kernel_sizes=[(3,3)]*4,
        strides=[(1,1),(2,2),(2,2),(2,2)],
        n_conv_per_stage=[2]*4, num_classes=1,
        n_conv_per_stage_decoder=[2,2,2],
        conv_bias=False, norm_op=nn.BatchNorm2d,
        norm_op_kwargs=dict(eps=1e-5, affine=True),
        dropout_op=None, dropout_op_kwargs=None,
        nonlin=nn.LeakyReLU, nonlin_kwargs=dict(inplace=False),
        deep_supervision=False
    )
    model = build_lkmunet_sde_film(cfg, strategy="stack").to(device).eval()
    B, M = 2, 3
    x_early = torch.randn(B, 1, 128, 128, device=device)
    x_mid   = torch.randn(B, M, 1, 128, 128, device=device)
    with torch.no_grad():
        y, *_ = model(x_early, t=1.0, mid_pet=x_mid)
    print("Output :", y.min().item(), y.max().item(), y.shape)  # 應在 [0,1] 內


In [ ]:
# Batch Visual & Curve
import os
from colorama import Fore, Style
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torchvision.models as models  # ✅ 修正 VGG19 import
from piq import multi_scale_ssim as calculate_msssim
import torch
import torch.nn as nn
import torch.nn.functional as F
from lpips import LPIPS  # ✅ 這是造成你之前 NameError 的根本原因

def visualize_batch(data, target, output, subjects, slices, save_path, epoch, fold):
    """
    Visualize input, ground truth, and model output for the first batch (6 images).
    - 使用 2×6 的 `subplot`
    - 影像逆時針旋轉 90°
    - 統一使用 `viridis` colormap
    - 只顯示 batch 內前 6 張影像
    """
    os.makedirs(save_path, exist_ok=True)
    num_images = min(6, data.shape[0])  # 確保 batch 不會超過 6 張

    fig, axes = plt.subplots(2, 6, figsize=(12, 4))  # 2×6 格狀顯示
    fig.suptitle(f"Fold {fold} | Epoch {epoch} | Batch Visualization", fontsize=14, fontweight="bold")

    # ✅ **如果 output 是 list，取最後一層**
    if isinstance(output, list):
        output = output[-1]  # 取最後一層作為視覺化

    for i in range(num_images):
        row, col = divmod(i, 6)  # 計算 subplot 位置

        # ✅ **確保 `target[i]` shape 為 (H, W)**
        gt_img = np.rot90(target[i].squeeze().cpu().numpy(), 1)  # `squeeze()` 移除單通道

        # ✅ **確保 `output[i]` shape 為 (H, W)**
        if i < output.shape[0]:  # 確保不超過 batch size
            syn_img = np.rot90(output[i].mean(dim=0).squeeze().cpu().numpy(), 1)  # `squeeze()` 移除多餘維度
        else:
            continue  # 如果 batch size 小於 6，跳過這個索引

        # GT (第一行)
        axes[0, col].imshow(gt_img, cmap="viridis")
        axes[0, col].set_title(f"GT: {subjects[i]} Slice {slices[i]}", fontsize=8)
        axes[0, col].axis("off")

        # Output (第二行)
        axes[1, col].imshow(syn_img, cmap="viridis")
        axes[1, col].set_title(f"Output: {subjects[i]} Slice {slices[i]}", fontsize=8)
        axes[1, col].axis("off")

    # 確保沒有多餘的空白 subplot
    for j in range(num_images, 6):  
        axes[0, j].axis("off")
        axes[1, j].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f"fold{fold}_epoch{epoch}_batch_visualization.png"))
    plt.close(fig)
# ✅ Loss Curve 與 Metric 曲線（即時更新）
def plot_curves(train_loss, valid_loss, psnr, ssim, save_path, fold, baseline_psnr, baseline_ssim):
    """
    繪製 Loss 和 Metric 曲線，並即時存儲 Loss curve 圖片（覆蓋前一張）。
    - 每個 epoch 更新 Loss 曲線
    - 存 numpy 供後續分析
    """
    os.makedirs(save_path, exist_ok=True)

    # ✅ 存為 numpy 供後續分析
    np.save(os.path.join(save_path, f"train_loss_fold_{fold}.npy"), np.array(train_loss))
    np.save(os.path.join(save_path, f"valid_loss_fold_{fold}.npy"), np.array(valid_loss))
    np.save(os.path.join(save_path, f"psnr_fold_{fold}.npy"), np.array(psnr))
    np.save(os.path.join(save_path, f"ssim_fold_{fold}.npy"), np.array(ssim))

    print(Fore.GREEN + f"[✔] Saved Loss and Metrics as .npy files in {save_path}" + Style.RESET_ALL)

    # ✅ 繪製 Loss 曲線（即時更新）
    plt.figure(figsize=(10, 5))
    sns.lineplot(x=range(len(train_loss)), y=train_loss, label="Train Loss", color="blue", linewidth=2.5)
    sns.lineplot(x=range(len(valid_loss)), y=valid_loss, label="Validation Loss", color="orange", linewidth=2.5)
    plt.title(f"Loss Curve (Fold {fold})", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f"loss_curve_fold_{fold}.png"))  # ✅ 覆蓋前一張 Loss curve
    plt.close()

    # ✅ PSNR 與 SSIM 曲線
    fig, ax1 = plt.subplots(figsize=(10, 5))
    color_psnr = "green"
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("PSNR", fontsize=12, color=color_psnr)
    ax1.plot(range(len(psnr)), psnr, label="PSNR", color=color_psnr, linewidth=2.5)
    ax1.axhline(y=baseline_psnr, color="green", linestyle="--", label="Baseline PSNR", linewidth=1.5)
    ax1.tick_params(axis="y", labelcolor=color_psnr)

    # SSIM 曲線
    ax2 = ax1.twinx()
    color_ssim = "red"
    ax2.set_ylabel("SSIM", fontsize=12, color=color_ssim)
    ax2.plot(range(len(ssim)), ssim, label="SSIM", color=color_ssim, linewidth=2.5)
    ax2.axhline(y=baseline_ssim, color="red", linestyle="--", label="Baseline SSIM", linewidth=1.5)
    ax2.tick_params(axis="y", labelcolor=color_ssim)

    fig.tight_layout()
    ax1.legend(loc="upper left", fontsize=10)
    ax2.legend(loc="upper right", fontsize=10)
    plt.title(f"Metrics Curve (Fold {fold})", fontsize=14, fontweight="bold")

    # ✅ 即時存儲
    plt.savefig(os.path.join(save_path, f"metrics_curve_fold_{fold}.png"))
    plt.close()

class HybridLoss_v4(nn.Module):
    def __init__(self, 
                 lambda_pixel=0.05,  
                 lambda_msssim=0.3,  
                 lambda_perceptual=0.2,  
                 lambda_style=0.0,  
                 lambda_edge=0.3,  
                 lambda_contrast=0.0,        #  Contrastive Learning
                 contrastive_type="triplet",  #  "cosine" or "triplet"
                 device="cuda:0"):
        super(HybridLoss_v4, self).__init__()
        self.lambda_pixel = lambda_pixel
        self.lambda_msssim = lambda_msssim
        self.lambda_perceptual = lambda_perceptual
        self.lambda_style = lambda_style
        self.lambda_edge = lambda_edge
        self.lambda_contrast = lambda_contrast
        self.device = device
        self.vgg = models.vgg19(pretrained=True).features[:16].eval().to(device)

        for param in self.vgg.parameters():
            param.requires_grad = False

        self.huber_loss = nn.SmoothL1Loss()
        self.lpips_loss_fn = LPIPS(net="vgg").to(device)
        self.contrastive_type = contrastive_type

        if contrastive_type == "triplet":
            self.contrastive_loss_fn = TripletContrastiveLoss(margin=0.5)
        else:
            self.contrastive_loss_fn = CosineContrastiveLoss(lambda_early=0.5, lambda_late=0.5)

    def gram_matrix(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, H * W)
        return torch.bmm(x, x.transpose(1, 2)) / (C * H * W)

    def laplacian_edge_loss(self, x):
        laplacian_kernel = torch.tensor(
            [[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
            dtype=torch.float32, device=x.device
        ).expand(x.shape[1], 1, 3, 3)
        edge_x = F.conv2d(x, laplacian_kernel, padding=1, groups=x.shape[1])
        return torch.mean(torch.abs(edge_x))

    def forward(self, outputs, targets, early_frame):
        if isinstance(outputs, list):
            outputs = outputs[-1]

        outputs = outputs.to(self.device)
        targets = targets.to(self.device)
        early_frame = early_frame.to(self.device)

        # 先補維度再 interpolate
        if targets.dim() == 3:
            targets = targets.unsqueeze(1)
        if early_frame.dim() == 3:
            early_frame = early_frame.unsqueeze(1)

        if outputs.shape != targets.shape:
            outputs = F.interpolate(outputs, size=targets.shape[2:], mode="bilinear", align_corners=False)

        outputs = torch.clamp(outputs, min=0, max=1)
        targets = torch.clamp(targets, min=0, max=1)
        early_frame = torch.clamp(early_frame, min=0, max=1)

        # outputs = outputs.mean(dim=1, keepdim=True)  # 移除 mean, 保持原 shape 做計算

        pixel_loss = self.huber_loss(outputs, targets)
        msssim_loss = 1 - calculate_msssim(outputs, targets, kernel_size=5, data_range=1.0, reduction='mean')
        outputs_vgg = self.vgg(outputs.repeat(1, 3, 1, 1))
        targets_vgg = self.vgg(targets.repeat(1, 3, 1, 1))
        perceptual_loss = self.huber_loss(outputs_vgg, targets_vgg)
        outputs_gram = self.gram_matrix(outputs_vgg)
        targets_gram = self.gram_matrix(targets_vgg)
        style_loss = F.mse_loss(outputs_gram, targets_gram)
        edge_loss = self.laplacian_edge_loss(outputs) + self.laplacian_edge_loss(targets)
        contrastive_loss = self.contrastive_loss_fn(outputs, early_frame, targets)

        total_loss = (
            self.lambda_pixel * pixel_loss +
            self.lambda_msssim * msssim_loss +
            self.lambda_perceptual * perceptual_loss +
            self.lambda_style * style_loss +
            self.lambda_edge * edge_loss +
            self.lambda_contrast * contrastive_loss
        )

        return total_loss
    
import lpips
import torch.nn.functional as F
from torchvision import models

class ImageFlowNetLoss(nn.Module):
    def __init__(self, lpips_weight=0.5, smoothness_weight=1e-4, contrastive_weight=1.0, device='cuda:0'):
        super().__init__()
        self.lpips_fn = LPIPS(net='vgg').to(device)
        self.lpips_weight = lpips_weight
        self.smoothness_weight = smoothness_weight
        self.contrastive_weight = contrastive_weight
        self.mse_loss = nn.MSELoss()

    def forward(self, outputs, targets, drift_field=None, early_latent=None, mid_latent=None):
        recon_loss = self.mse_loss(outputs, targets)

        outputs_rgb = outputs.expand(-1, 3, -1, -1)
        targets_rgb = targets.expand(-1, 3, -1, -1)
        lpips_loss = self.lpips_fn(outputs_rgb, targets_rgb).mean()

        total_loss = recon_loss + self.lpips_weight * lpips_loss

        # Smoothness loss
        if drift_field is not None:
            smoothness_loss = drift_field.pow(2).mean()
            total_loss += self.smoothness_weight * smoothness_loss

        # Contrastive loss
        if early_latent is not None and mid_latent is not None:
            early_norm = F.normalize(early_latent, p=2, dim=1)
            mid_norm = F.normalize(mid_latent, p=2, dim=1)
            contrastive_loss = 1.0 - (early_norm * mid_norm).sum(dim=1).mean()
            total_loss += self.contrastive_weight * contrastive_loss

        return total_loss

# 0429 被改
class ImageFlowNetLoss(nn.Module):
    def __init__(self, smooth_w=1e-4, cont_w=1.):
        super().__init__()
        self.mse  = nn.MSELoss()
        self.w_sm = smooth_w
        self.w_c  = cont_w
    def forward(self, pred, tgt, drift=None, early=None, mid=None):
        loss = self.mse(pred, tgt)
        if drift is not None:
            loss += self.w_sm * drift.pow(2).mean()
        if (early is not None) and (mid is not None):
            e = F.normalize(early, p=2, dim=1)
            m = F.normalize(mid,   p=2, dim=1)
            loss += self.w_c * (1 - (e*m).sum(1).mean())
        return loss


def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2) + 1e-10
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

def adjust_mid_pet_shape(z):
    z = z.squeeze()
    if z.ndim == 3:
        z = z.unsqueeze(0).unsqueeze(2)
    elif z.ndim == 4:
        z = z.unsqueeze(0)
    elif z.ndim != 5:
        raise ValueError(f"[adjust_mid_pet_shape] Unexpected shape: {z.shape}")
    return z

def visualize_batch(data, target, output, subjects, slices, save_path, epoch, fold):
    os.makedirs(save_path, exist_ok=True)
    num_images = min(6, data.shape[0])
    fig, axes = plt.subplots(2, 6, figsize=(12, 4))
    fig.suptitle(f"Fold {fold} | Epoch {epoch} | Batch Visualization", fontsize=14, fontweight="bold")

    if isinstance(output, list):
        output = output[-1]

    for i in range(num_images):
        row, col = divmod(i, 6)
        gt_img = np.rot90(target[i].squeeze().cpu().numpy(), 1)
        syn_img = np.rot90(output[i].mean(dim=0).squeeze().cpu().numpy(), 1)
        axes[0, col].imshow(gt_img, cmap="viridis")
        axes[0, col].set_title(f"GT: {subjects[i]} Slice {slices[i]}", fontsize=8)
        axes[0, col].axis("off")
        axes[1, col].imshow(syn_img, cmap="viridis")
        axes[1, col].set_title(f"Output: {subjects[i]} Slice {slices[i]}", fontsize=8)
        axes[1, col].axis("off")
    for j in range(num_images, 6):
        axes[0, j].axis("off")
        axes[1, j].axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f"fold{fold}_epoch{epoch}_batch_visualization.png"))
    plt.close(fig)

class TripletContrastiveLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()  # ✅ 更簡潔的寫法
        self.margin = margin

    def forward(self, output, early_frame, late_frame):
        # 計算 L2 距離
        positive_dist = F.mse_loss(output, late_frame)  # Output 應該接近 Late Frame
        negative_dist = F.mse_loss(output, early_frame)  # Output 應該遠離 Early Frame

        # Triplet Loss 計算
        loss = F.relu(positive_dist - negative_dist + self.margin)
        return loss
    
class CosineContrastiveLoss(nn.Module):
    def __init__(self, lambda_early=0.5, lambda_late=0.5):
        super().__init__()
        self.lambda_early = lambda_early
        self.lambda_late = lambda_late

    def forward(self, output, early_frame, late_frame):
        # **確保輸出的通道數匹配 early_frame**
        if output.shape[1] == 1:
            output = output.repeat(1, 3, 1, 1)

        # **確保 output 和 late_frame 解析度一致**
        if output.shape[2:] != late_frame.shape[2:]:
            output = F.interpolate(output, size=late_frame.shape[2:], mode="bilinear", align_corners=False)

        if early_frame.shape[2:] != late_frame.shape[2:]:
            early_frame = F.interpolate(early_frame, size=late_frame.shape[2:], mode="bilinear", align_corners=False)

        cos_sim_late = cosine_loss(output, late_frame)
        cos_sim_early = cosine_loss(output, early_frame)

        loss = self.lambda_late * (1 - cos_sim_late) + self.lambda_early * cos_sim_early
        return loss

In [ ]:
# -*- coding: utf-8 -*- Training & Valudate (GuideDropOut)
"""
train_loop_micro_ckpt.py  –  LKMUNet-SDE-FiLM    (micro-batch, ckpt, LPIPS val-only)
===================================================================================
"""

from __future__ import annotations    
from pathlib import Path
from typing import  Tuple, Dict, List
import gc, math, os, json
import numpy as np              
import pandas as pd              
import matplotlib.pyplot as plt  
import seaborn as sns            

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.checkpoint import checkpoint
from torchmetrics.functional import peak_signal_noise_ratio as calc_psnr
from torchmetrics.functional import structural_similarity_index_measure as calc_ssim
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
from lpips import LPIPS
from colorama import Fore, Style
from tqdm import tqdm
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import random    
sns.set_style("whitegrid")

# ────────────────────────── 全域開關 ──────────────────────────
PLOT_METRIC = True     # False: 不存 .npy/.png
VISUALIZE   = True     # False: 不產 validation 圖
SAVE_SYMLINK= True     # False: 不建立 best.pth 的 symbolic link
ACC_STEPS    = 4   
LR                = 1e-4               # ★
WEIGHT_DECAY      = 3e-2                
DROP_FULL_P      = 0.3          # 整批拿掉 guide 的機率
DROP_PARTIAL_P   = 0.3          # 在「保留情況」下，每張 guide 被丟掉的機率
CLIP_GRAD_NORM   = 1.0          # 為了穩定 ODE，建議 1.0
# ─────────────────────────────────────────────────────────────

# ═══════════════════════ Utilities ════════════════════════
def ensure_5d(t: torch.Tensor) -> torch.Tensor:
    t = t.squeeze()
    if t.ndim == 5: return t
    if t.ndim == 4: return t.unsqueeze(2)
    if t.ndim == 3: return t.unsqueeze(1).unsqueeze(2)
    raise ValueError(f"latent shape {t.shape} unsupported")

def squeeze_early(x: torch.Tensor) -> torch.Tensor:
    return x.squeeze(2) if (x.ndim == 5 and x.size(2) == 1) else x

# ────────────────────────── Baseline CSV ───────────────────
def load_baseline_metrics(
        fold:int,
        csv_path:str="/home/allen/SAM2-UNet-main/all_metrics_results_0418.csv"
) -> Dict[str, float]:
    """Return {'psnr':…, 'ssim':…, 'lpips':…} for given fold."""
    df = pd.read_csv(csv_path)
    row = df[df["Fold"] == fold]
    if row.empty:
        return {'psnr':-np.inf, 'ssim':-np.inf, 'lpips':np.inf}
    return {'psnr': row["mean_psnr"].max(),
            'ssim': row["mean_ssim"].max(),
            'lpips': row["mean_lpips"].min()}

# ────────────────────────── helper：metric plot ────────────────
def update_and_plot_metrics(
        epoch:int,
        train_losses:List[float], valid_losses:List[float],
        psnr_scores:List[float],  ssim_scores:List[float],
        lpips_scores:List[float], mse_scores:List[float],
        save_dir:Path):

    loss_dir   = save_dir / "Loss";   loss_dir.mkdir(parents=True, exist_ok=True)
    metric_dir = save_dir / "Metric"; metric_dir.mkdir(parents=True, exist_ok=True)

    # ---- 保存 numpy (覆蓋) -----------------------------
    np.save(loss_dir / "train_losses.npy",  np.array(train_losses,  np.float32))
    np.save(loss_dir / "valid_losses.npy",  np.array(valid_losses,  np.float32))
    np.save(metric_dir / "psnr_scores.npy", np.array(psnr_scores, np.float32))
    np.save(metric_dir / "ssim_scores.npy", np.array(ssim_scores, np.float32))
    np.save(metric_dir / "lpips_scores.npy",np.array(lpips_scores,np.float32))
    np.save(metric_dir / "mse_scores.npy",  np.array(mse_scores,  np.float32))

    # ---- 畫 Loss (同圖) --------------------------------
    plt.figure(figsize=(6,4))
    sns.lineplot(x=range(1,len(train_losses)+1), y=train_losses, label="Train")
    sns.lineplot(x=range(1,len(valid_losses)+1), y=valid_losses, label="Valid")
    best = int(np.argmin(valid_losses)); best_val = valid_losses[best]
    plt.scatter(best+1, best_val, color='red')
    plt.annotate(f"Best E{best+1}: {best_val:.4f}", (best+1,best_val),
                 xytext=(0,-12), textcoords="offset points", ha="center", fontsize=8)
    plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.tight_layout(); plt.savefig(loss_dir/"loss_curve.png", dpi=300); plt.close()

    # ---- 單指標曲線 ------------------------------------
    def _curve(arr, name, larger_better=True):
        plt.figure(figsize=(6,4))
        sns.lineplot(x=range(1,len(arr)+1), y=arr)
        idx = int(np.argmax(arr) if larger_better else np.argmin(arr))
        plt.scatter(idx+1, arr[idx], color='red')
        plt.annotate(f"Best E{idx+1}: {arr[idx]:.4f}", (idx+1, arr[idx]),
                     xytext=(0,-12), textcoords="offset points",
                     ha="center", fontsize=8)
        plt.title(f"{name} Curve"); plt.xlabel("Epoch"); plt.ylabel(name)
        plt.tight_layout()
        plt.savefig(metric_dir/f"{name.lower()}_curve.png", dpi=300)
        plt.close()
    _curve(psnr_scores,"PSNR",True)
    _curve(ssim_scores,"SSIM",True)
    _curve(lpips_scores,"LPIPS",False)
    _curve(mse_scores,"MSE",False)

def to_5d(x: torch.Tensor | None) -> torch.Tensor | None:
    """
    將 latent tensor 轉成 [B,M,C,H,W]：
    - 支援 shape:
        [B,M,C,H,W] (原生 5‑D)
        [B,C,M,H,W] (C、M 掉換)
        [B,1,M,1,H,W] 或 [B,1,M,H,W]  (常見 dataset layout)
        [B,C,H,W] / [B,M,C,H,W]         (回退處理)
    """
    if x is None:
        return None

    # ------------- 5‑D 已是我們要的 -----------------
    if x.ndim == 5:
        # 若是 [B,C,M,H,W] -> 調成 [B,M,C,H,W]
        if x.shape[2] in (1, 3) and x.shape[1] != 1:  # 判斷誰是 C
            x = x.permute(0, 2, 1, 3, 4).contiguous()
        return x

    # ------------- 6‑D 常見: [B,1,M,1,H,W] ----------
    if x.ndim == 6:
        B, C1, M, C2, H, W = x.shape
        if C1 == 1 and C2 == 1:
            return x.view(B, M, 1, H, W)             # → [B,M,C=1,H,W]
        if C2 == 1:                                  # [B,C,M,1,H,W] 想保留 C
            return x.permute(0, 2, 1, 4, 5).contiguous()
        # 其他 layout 視需求再補
        raise ValueError(f"6‑D latent 不支援形狀 {x.shape}")

    # ------------- 4‑D / 3‑D：當作只有一張 guide ----
    if x.ndim == 4:  # [B,C,H,W]
        return x.unsqueeze(1)                        # [B,1,C,H,W]
    if x.ndim == 3:  # [B,H,W]
        return x.unsqueeze(1).unsqueeze(2)           # [B,1,1,H,W]

    raise ValueError(f"unsupported latent shape {x.shape}")


def guide_dropout(mid_5d: torch.Tensor | None,
                  p_full: float = DROP_FULL_P,
                  p_part: float = DROP_PARTIAL_P) -> torch.Tensor | None:  ### ★ NEW
    if mid_5d is None:                      # 本來就沒 guide
        return None
    if torch.rand(1) < p_full:              # ❶ 整批拔掉
        return None
    # ❷ 部分拔掉
    if mid_5d.size(1) > 1:
        keep = torch.rand(mid_5d.size(1), device=mid_5d.device) > p_part
        if keep.sum() == 0:                 # 至少留一張
            keep[torch.randint(0, keep.numel(), (1,))] = True
        mid_5d = mid_5d[:, keep]
    return mid_5d                           # 仍然是 5‑D


# ──────────────────── Validation Visualization ──────────────────
def visualize_validation_epoch(
    inputs: Dict[str, np.ndarray],
    guides: Dict[str, np.ndarray],
    synths: Dict[str, np.ndarray],
    gts:    Dict[str, np.ndarray],
    epoch:  int,
    save_dir: Path,
    json_path: str = "/home/allen/SAM2-UNet-main/test_subjects_info_flat.json",
    dpi: int = 300
):
    if not inputs:
        return

    vis_dir = save_dir / "Visualizations"
    vis_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path, "r") as f:
        data = json.load(f)
    meta = {d["Subject"]: d for d in data} if isinstance(data, list) else data

    prefer     = ["Subject010","Subject042","Subject001","Subject036","Subject076"]
    diag_order = ["AD","CAA_ICH","CAA_CI","HTN"]
    chosen: Dict[str,str] = {}
    for diag in diag_order:
        cand = [s for s in prefer if s in inputs and meta.get(s, {}).get("Diagnosis") == diag]
        if not cand:
            cand = [s for s in inputs if meta.get(s, {}).get("Diagnosis") == diag]
        if cand:
            chosen[diag] = cand[0]
    if not chosen:
        return

    rows = len(chosen)
    fig, ax = plt.subplots(rows, 6, figsize=(18, 3*rows), dpi=dpi)
    if rows == 1:
        ax = [ax]
    titles = ["Input", "Guide-1", "Guide-2", "Guide-3", "Synthesis", "GT"]

    for r, diag in enumerate(diag_order):
        if diag not in chosen:
            continue
        sid = chosen[diag]
        row = ax[r]
        imgs = [inputs[sid], *guides[sid], synths[sid], gts[sid]]

        for c, img in enumerate(imgs):
            im = np.rot90(img)
            vmin, vmax = im.min(), im.max()
            row[c].imshow(im, cmap="hot", vmin=vmin, vmax=vmax)
            row[c].axis("off")
            row[c].set_title(f"{titles[c]}\nmin={vmin:.3f}, max={vmax:.3f}", fontsize=9)

        info = meta.get(sid, {})
        row[0].set_ylabel(
            f"{sid}\n{diag}\nAβ:{info.get('Abeta', '?')}",
            fontsize=10, rotation=0, labelpad=50, va="center"
        )

    plt.suptitle(f"Validation Visualization – Epoch {epoch}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(vis_dir / f"epoch_{epoch:03d}_validation.png")
    plt.close()

# ═══════════════════════  Core components  ════════════════════════════════════════════

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
warmup_sched = lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda it: min(1., it / 2000))

# 定義 ReduceLROnPlateau

plateau_sched = lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=10,
    threshold=1e-4,
    verbose=True
)

# ───────────────────────────────── loss ───────────────────────
   
class ImageFlowNetLoss(nn.Module):
    """
    L = wm * MSE + ws * (1 - SSIM)/2 + wl * LPIPS
        + w_sm * || drift ||² + w_c * (1 - cos_sim(early, mid))

    其中 (1-SSIM)/2 可把 SSIM ∈ [-1,1] 映射到 [0,1]，和 MSE 量級一致；
    LPIPS 已經是距離形式 (越小越好)。
    """
    def __init__(
        self,
        mse_w: float   = 1.0,
        ssim_w: float  = 1.0,
        lpips_w: float = 1.0,
        smooth_w: float = 5e-4,
        cont_w: float   = 1.0,
        resize_lpips: bool = True        # LPIPS 需輸入 3-channel；此選項會自動複製通道
    ):
        super().__init__()
        self.mse_w   = mse_w
        self.ssim_w  = ssim_w
        self.lpips_w = lpips_w
        self.w_sm    = smooth_w
        self.w_c     = cont_w

        # torchmetrics 的 SSIM 要在 forward 用；LPIPS 建一個 once-for-all 的 model
        self.mse   = nn.MSELoss()
        self.lpips = LPIPS(net='vgg').eval()          # 記得在外部 model.eval()
        for p in self.lpips.parameters():             # LPIPS 不更新梯度
            p.requires_grad_(False)
        self.resize_lpips = resize_lpips

    @staticmethod
    def _to_rgb(x: torch.Tensor) -> torch.Tensor:
        """[B,1,H,W] → [B,3,H,W]  (通道複製；LPIPS 需要 3 通道)"""
        return x.repeat(1, 3, 1, 1)

    def forward(
        self,
        pred:  torch.Tensor,
        tgt:   torch.Tensor,
        drift: torch.Tensor | None = None,
        early: torch.Tensor | None = None,
        mid:   torch.Tensor | None = None,
    ) -> torch.Tensor:

        # ---------- 基本影像誤差 -------------------------------------------------
        mse  = self.mse(pred, tgt)
        ssim = (1.0 - ssim_fn(pred, tgt, data_range=1.0)) * 0.5     # → [0,1]
        if self.resize_lpips and pred.size(1) == 1:
            lpips_val = self.lpips(self._to_rgb(pred), self._to_rgb(tgt)).mean()
        else:
            lpips_val = self.lpips(pred, tgt).mean()

        loss = (
            self.mse_w   * mse  +
            self.ssim_w  * ssim +
            self.lpips_w * lpips_val
        )

        # ---------- 其他正則項 ---------------------------------------------------
        if drift is not None:
            loss = loss + self.w_sm * drift.pow(2).mean()

        if (early is not None) and (mid is not None):
            e = F.normalize(early, p=2, dim=1)
            m = F.normalize(mid,   p=2, dim=1)
            loss = loss + self.w_c * (1.0 - (e * m).sum(1).mean())

        return loss

# ───────────────────────────────── early-stopper ─────────────────────────
class EarlyStopper:
    """Early-stop when composite score (psnr+25*ssim) OR lpips no longer improves."""
    def __init__(self, patience:int, delta:float,
                 baseline:Dict[str,float]):
        self.patience, self.delta = patience, delta
        self.best_score   = -np.inf      # higher is better
        self.best_lpips   = np.inf
        self.wait = 0
        self.baseline = baseline
        self.first = True

    def check_improve(self, psnr:float, ssim:float, lpips:float) -> bool:
        if self.first:                  # ★ 初始一定回 True
            self.first = False
            return True
        better_than_base = (psnr>self.baseline['psnr'] and
                            ssim>self.baseline['ssim'] and
                            lpips<self.baseline['lpips'])
        if not better_than_base:
            return False
        score = psnr + 25*ssim
        improved = (score > self.best_score + self.delta) or (lpips < self.best_lpips - self.delta)
        if improved:
            self.best_score, self.best_lpips, self.wait = score, lpips, 0
        else:
            self.wait += 1
        return improved

    @property
    def stop(self)->bool: return self.wait >= self.patience

# ───────────────────────────────── ckpt 包裝 ─────────────────────────────
class CheckpointWrapper(nn.Module):
    """把 module 用 torch.utils.checkpoint 包起來"""
    def __init__(self, mod: nn.Module): super().__init__(); self.mod=mod
    def forward(self, x):            return checkpoint(self.mod, x)

def apply_gradient_checkpoint(model: nn.Module):
    """遞迴尋找 BiPixel/WindowMambaLayer 並替換"""
    for name, child in list(model.named_children()):
        if child.__class__.__name__ in ("BiPixelMambaLayer", "BiWindowMambaLayer"):
            setattr(model, name, CheckpointWrapper(child))
        else:
            apply_gradient_checkpoint(child)

# ═══════════════════════ Train & Validate ══════════════════════
def train_and_validate(model          : nn.Module,
                       train_loader   : DataLoader,
                       valid_loader   : DataLoader,
                       optimizer,
                       warmup_sched,         
                       epoch_sched,           
                       num_epochs     : int,  
                       save_path      : str|Path,
                       fold           : int,
                       acc_steps      : int = ACC_STEPS, 
                       baseline_csv   : str="/home/allen/SAM2-UNet-main/all_metrics_results_0418.csv"):

    device   = next(model.parameters()).device
    criterion_tr = ImageFlowNetLoss().to(device)
    lpips_val    = LPIPS(net='vgg').to(device).eval()     # 只 val 用

    save_path = Path(save_path)
    baseline = load_baseline_metrics(fold, baseline_csv)
    stopper  = EarlyStopper(patience=25, delta=0.05, baseline=baseline)

    ckpt_dir = save_path/f"fold_{fold}/ckpt"; ckpt_dir.mkdir(parents=True,exist_ok=True)

   # -------- histories --------
    h_tr, h_val = [], []
    h_psnr, h_ssim, h_lpips, h_mse = [], [], [], []

    for epoch in range(1, num_epochs+1):
        
        # ---------- reset epoch‑level meters ----------
        p_full = min(1.0, 0.3 + epoch / 40) 
        train_mse_sum = 0.0
        nsamp_train   = 0
        # -------------------- TRAIN ----------------------------------------
        model.train(); running=0; step=0
        optimizer.zero_grad(set_to_none=True)

        for batch in tqdm(train_loader, desc=f'E{epoch:02d}[train]'):
            x   = batch['input'].float().to(device)
            tgt = batch['ground_truth'].float().to(device)
            #mid = ensure_5d(batch['latent_target'].float().to(device))
            mid = to_5d(batch["latent_target"].float().to(device))
            x, tgt = squeeze_early(x), squeeze_early(tgt)

            # ★ Guide Drop‑out
            mid = guide_dropout(mid,p_full = p_full)

            pred, drift, early_lat, mid_lat = model(x, t=torch.tensor(1.0,device=device), mid_pet=mid)
            loss = criterion_tr(pred,tgt,drift,early_lat,mid_lat)/acc_steps
            loss.backward()

            mse_b = F.mse_loss(pred, tgt).item() * x.size(0)
            train_mse_sum += mse_b
            nsamp_train   += x.size(0)

            step += 1
            if step % acc_steps == 0:
                nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
                optimizer.step(); 
                optimizer.zero_grad(set_to_none=True)
                # -------- warm-up 每一步更新 --------
                warmup_sched.step()

            running += loss.item() * acc_steps   # 還原至真實 loss
        train_loss = running/len(train_loader)
        train_mse  = train_mse_sum / nsamp_train

        # ============= Validation ========================
        model.eval()
        psnr=ssim=lp=mse_sum=val_loss_sum=0; nsamp=0
        vis_inputs,vis_guides,vis_syns,vis_gts={}, {}, {}, {}

        with torch.no_grad():
            for batch in tqdm(valid_loader, desc=f'E{epoch:02d}[val]'):
                 # ● 以 dataset 提供的 'subject' 為唯一識別
                sid_list = [s for s in batch.get('subject')]       # ← list[str]

                x   = batch['input'        ].float().to(device)
                tgt = batch['ground_truth' ].float().to(device)
                #mid = ensure_5d(batch['latent_target'].float().to(device))   # [B,M,C,H,W]
                mid = to_5d(batch["latent_target"].float().to(device))
                x, tgt = squeeze_early(x), squeeze_early(tgt)

                pred, drift, early_lat, mid_lat = model(
                    x, t=torch.tensor(1.0, device=device), mid_pet=None)
                bs=x.size(0); nsamp+=bs
                # --------- ❶ ImageFlowNet valid loss ------------
                val_loss_sum += criterion_tr(
                        pred, tgt, drift, early_lat, mid_lat).item() * bs
                psnr += calc_psnr(pred,tgt,data_range=1.).item()*bs
                ssim += calc_ssim(pred.clamp(0,1),tgt.clamp(0,1),data_range=1.).item()*bs
                lp   += lpips_val(pred.expand(-1,3,-1,-1),
                                  tgt.expand(-1,3,-1,-1)).mean().item()*bs
                mse_sum += F.mse_loss(pred,tgt).item()*bs
                
                 # ----- 影像收集 (僅在開啟 VISUALIZE 時) -----------------
                if VISUALIZE:
                    x_np    = x.cpu().numpy()              # [B,1,H,W]
                    pred_np = pred.cpu().numpy()           # [B,1,H,W]
                    tgt_np  = tgt.cpu().numpy()            # [B,1,H,W]
                    g_np    = mid.cpu().numpy()            # [B,M,1,H,W]

                    for i, sid in enumerate(sid_list):
                        if sid not in vis_inputs:
                            vis_inputs[sid] = x_np[i, 0]               # (H,W)
                            # 取前 3 張 guide，若不足 3 張就重複最後一張填滿
                            g3 = g_np[i, :, 0]                         # (M,H,W)
                            if g3.shape[0] < 3:
                                g3 = np.pad(g3, ((0,3-g3.shape[0]),(0,0),(0,0)),
                                            mode='edge')
                            vis_guides[sid] = g3[:3]                   # (3,H,W)
                            vis_syns[sid]   = pred_np[i, 0]
                            vis_gts[sid]    = tgt_np[i, 0]
                            
        psnr/=nsamp; ssim/=nsamp; lp/=nsamp; mse = mse_sum/nsamp
        val_loss = val_loss_sum / nsamp
        train_mse = train_mse_sum / nsamp_train          # ← 看下一節
        val_mse   = mse_sum       / nsamp

        print(  f"E{epoch:02d} ▸ "
                f"train‑L {train_loss:.4f} | "
                f"valid‑L {val_loss:.4f} | "
                f"train‑MSE {train_mse:.5f} | "
                f"val‑MSE {val_mse:.5f} | "
                f"PSNR {psnr:.2f}  SSIM {ssim:.4f}  LPIPS {lp:.4f}")
                    
        # 用 validation 的 SSIM 來驅動 ReduceLROnPlateau
        #plateau_sched.step(ssim)
        epoch_sched.step(ssim)

        # --------  histories -----------------------------
        h_tr.append(train_loss); 
        h_val.append(val_loss)
        h_psnr.append(psnr); 
        h_ssim.append(ssim); 
        h_lpips.append(lp); 
        h_mse.append(val_mse) 

        if PLOT_METRIC:
            update_and_plot_metrics(epoch,h_tr,h_val,h_psnr,h_ssim,h_lpips,h_mse,save_path)
        if VISUALIZE:
            visualize_validation_epoch(vis_inputs,vis_guides,vis_syns,vis_gts,
                                       epoch,save_path)

        # -------- Early-stop & Save-best ---------------------------------
        improved = stopper.check_improve(psnr, ssim, lp)

        if improved:
            # —— 1. 依指標組檔名（無論是不是第一次） ——————————
            ckpt_name = (
                f"fold{fold}_epoch{epoch:03d}"
                f"_psnr{psnr:.2f}_ssim{ssim:.4f}_lpips{lp:.4f}.pth"
            )
            ckpt_path = ckpt_dir / ckpt_name

            torch.save(
                {"epoch": epoch,
                "model": model.state_dict(),
                "opt":   optimizer.state_dict()},
                ckpt_path,
            )

            # —— 2. 更新 / 建立 best.pth 連結 ————————————————
            if SAVE_SYMLINK:
                best_link = ckpt_dir / "best.pth"
                try:
                    if best_link.exists() or best_link.is_symlink():
                        best_link.unlink()
                    # 用 *完整路徑* 當 src，避免相對路徑失效
                    os.symlink(src=str(ckpt_path), dst=str(best_link))
                except OSError as e:
                    print(Fore.YELLOW + f"[symlink] {e}" + Style.RESET_ALL)

            print(Fore.GREEN + "  ↑ new best – model saved" + Style.RESET_ALL)

        # -------- Always save last-epoch checkpoint ----------------------
        if epoch == num_epochs:
            last_name = (
                f"fold{fold}_epoch{epoch:03d}"
                f"_psnr{psnr:.2f}_ssim{ssim:.4f}_lpips{lp:.4f}_last.pth"
            )
            last_path = ckpt_dir / last_name
            torch.save(
                {"epoch": epoch,
                "model": model.state_dict(),
                "opt":   optimizer.state_dict()},
                last_path,
            )

            # 建立 last.pth shortcut，方便快速載入「最新權重」
            last_link = ckpt_dir / "last.pth"
            try:
                if last_link.exists() or last_link.is_symlink():
                    last_link.unlink()
                os.symlink(src=str(last_path), dst=str(last_link))
            except OSError as e:
                print(Fore.YELLOW + f"[symlink] {e}" + Style.RESET_ALL)

            print(Fore.CYAN + f"  → final‑epoch model saved to {last_path.name}" + Style.RESET_ALL)

torch.cuda.empty_cache(); gc.collect()

# ──────────────────────────── Example usage ──────────────────────────────
if __name__ == "__main__":
    # 0. 構建 model  (請自行 import 您的 builder)
    # model = build_lkmunet_sde_film(cfg, strategy="auto").to("cuda:0")
    # ——> 這裡示範：
    # from model import build_lkmunet_sde_film, cfg
    # model = build_lkmunet_sde_film(cfg).to("cuda:0")

    # ★ 加 gradient-checkpoint (一行即可)
    # apply_gradient_checkpoint(model)

    # 1. DataLoader / Opt / Sched 自行準備 …
    pass


In [ ]:
# -*- coding: utf-8 -*- Five Fold Version
"""
train.py – LKMUNet-SDE-FiLM  (SDE 版，可改 ODE 版 builder)
改動項目：
1.  base-LR 5e-5
2.  2 k iter linear warm-up + CosineWarmRestarts(T0=30,T_mult=2)
3.  gradient-accumulation 步數 ACC_STEPS = 4
4.  對 train_and_validate 傳入 sched_tuple
"""

import os, random, warnings, gc, json
from datetime import datetime
from pathlib import Path
from colorama import Fore, Style
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ────────────── 全域參數 ─────────────────────────────────────────
SEEDS            = [42]          # ← ❶ 要跑的兩個 seed   /42 /2025 /31415 / 1105
FOLDS            = [0,1,2,3,4]        # ← ❷ 仍舊五個 fold
NUM_EPOCHS        = 150
BATCH_SIZE        = 16
ACC_STEPS         = 4                  # ★
RESIZE_HW         = (128, 128)
DATA_ROOT         = "./Preprocessed_Early2Late_withLatent_0729"
BASE_PATH         = "./2025_0826_ImageFlowNetLoss(SSIM_enh_LPS_enh)_Fold0-4_LKMUNet_SDE(1e-4)_Stack_plateau_GuideDp0.3"
LR                = 1e-4            # ★
WEIGHT_DECAY      = 3e-2
T0                = 30                 # CosineWarmRestarts 首週期
STRATEGY          = "stack"             # "auto" | "concat" | "stack"
DEVICE            = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
warnings.filterwarnings("ignore")

# ───────────── reproducibility ─────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(Fore.GREEN+f"[Seed] {seed} fixed"+Style.RESET_ALL)

def make_loader(split:str, fold:int):
    ds = Early2LateWithLatentDataset(
            pet_dir=DATA_ROOT,
            dataset_type=split,
            fold=fold,
            resize_image_size=RESIZE_HW,
            split=split_all)
    return DataLoader(
        ds, batch_size=BATCH_SIZE,
        shuffle=(split=="train"),
        num_workers=4, pin_memory=True
    )

# ─────────────  model config  ─────────────────────────────────
cfg = dict(
    input_channels=1,
    n_stages=4,
    features_per_stage=[32,64,128,256],
    conv_op=nn.Conv2d,
    kernel_sizes=[(3,3)]*4,
    strides=[(1,1),(2,2),(2,2),(2,2)],
    n_conv_per_stage=[2]*4,
    num_classes=1,
    n_conv_per_stage_decoder=[2,2,2],
    deep_supervision=False,
    conv_bias=False,
    norm_op=nn.BatchNorm2d,
    norm_op_kwargs=dict(eps=1e-5, affine=True),
    dropout_op=None,
    dropout_op_kwargs=None,
    nonlin=nn.LeakyReLU,
    nonlin_kwargs=dict(inplace=False),
)
# ---------- 2. 把「單一 fold 的全部流程」包進函式 ----------

def run_one_fold(fold:int):
    print(Fore.CYAN + f"\n=== Train Fold {fold} ===" + Style.RESET_ALL)

    # ─────────────  建立模型  ─────────────────────────────────────
    model = build_lkmunet_sde_film(cfg, strategy=STRATEGY).to(DEVICE) # build_lkmunet_ode_film / build_lkmunet_sde_film
    apply_gradient_checkpoint(model)                                  # 若要 ODE 版自行替換 builder

    # ─────────────  DataLoader  ──────────────────────────────────
    train_loader  = make_loader("train", fold)
    valid_loader  = make_loader("val",   fold)

    # ─────────────  Optimizer & 兩段 Scheduler  ──────────────────
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # ① linear warm-up（前 2000 iter 線性升到 LR）
    warm_iters   = 2_000
    warmup_fn    = lambda it: min(1., it / warm_iters)
    warmup_sched = lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_fn)

    # ② validation-plateau learning rate
    plateau_sched = lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',            # SSIM 越大越好
        factor=0.5,            # 每次 LR × 0.5
        patience=10,           # 如果 8 個 epoch SSIM 都沒提升就降 LR
        threshold=1e-4,        # SSIM 最小改善門檻
        cooldown=5,            # 降完之後過 5 個 epoch 才再觀察
        verbose=True,
    )

    # ─────────────  儲存路徑  ────────────────────────────────────
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fold_dir  = Path(BASE_PATH) / f"LKMUSDE_F{fold}_{timestamp}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # ─────────────  執行訓練  ────────────────────────────────────
    train_and_validate(
        model,
        train_loader,
        valid_loader,
        optimizer,
        warmup_sched,      # step-level scheduler
        epoch_sched=plateau_sched,     # validation-level scheduler
        num_epochs=NUM_EPOCHS,        # num_epochs
        save_path=fold_dir,
        fold=fold,
        acc_steps=ACC_STEPS
    )

    torch.cuda.empty_cache(); gc.collect()

# ---------- 3. 主程式：依序跑 5 folds ----------

if __name__ == "__main__":
    for sd in SEEDS:                 # ← ❸ 先換 seed
        set_seed(sd)                 #    （一定要在每輪 loop 內呼叫）
        # ▸ 讓輸出路徑帶 seed，資料互不覆蓋
        BASE_PATH = f"./runs_seed_LKMUNet_SDE{sd:05d}"
        for f in FOLDS:              #    再跑 5 個 fold
            run_one_fold(f)

    print(Fore.GREEN+"All seeds & folds finished!"+Style.RESET_ALL)


In [ ]:
# inference_multi_folds.py  – LKMUNet-SDE-FiLM  (兩個 fold 一次跑)
import gc, re, warnings
from pathlib import Path
from typing import List, Dict

import torch
from colorama import Fore, Style
from lpips import LPIPS
from torch.utils.data import DataLoader
from torchmetrics.functional import (
    peak_signal_noise_ratio as psnr_fn,
    structural_similarity_index_measure as ssim_fn,
)
from tqdm import tqdm
import numpy as np
import pandas as pd

# ---------- 0. 參數 -----------------------------------------------------
from pathlib import Path

#Original + LPS_enh
SDE_DIRS = [
    "/disk/disk4/allen/SAM2-UNet-main/runs_seed_LKMUNet_SDE00042/LKMUSDE_F0_20250729_171236", 
    "/disk/disk4/allen/SAM2-UNet-main/runs_seed_LKMUNet_SDE00042/LKMUSDE_F1_20250729_191445",
    "/disk/disk4/allen/SAM2-UNet-main/runs_seed_LKMUNet_SDE00042/LKMUSDE_F2_20250729_212710",
    "/disk/disk4/allen/SAM2-UNet-main/runs_seed_LKMUNet_SDE00042/LKMUSDE_F3_20250729_234517",
    "/disk/disk4/allen/SAM2-UNet-main/runs_seed_LKMUNet_SDE00042/LKMUSDE_F4_20250730_020742"
]

# SDE_DIRS = [
#     "/disk/disk4/allen/SAM2-UNet-main/runs_seed00042/LKMUSDE_F0_20250701_185354", 
#     "/disk/disk4/allen/SAM2-UNet-main/runs_seed00042/LKMUSDE_F1_20250628_223903",
#     "/disk/disk4/allen/SAM2-UNet-main/runs_seed00042/LKMUSDE_F2_20250629_005804",
#     "/disk/disk4/allen/SAM2-UNet-main/runs_seed00042/LKMUSDE_F3_20250629_032015",
#     "/disk/disk4/allen/SAM2-UNet-main/runs_seed00042/LKMUSDE_F4_20250629_054508"
# ]

# 如果要跑 SDE，選 SDE_DIRS；要跑 ODE，選 ODE_DIRS
TARGET_DIRS = SDE_DIRS  # 或 ODE_DIRS

# 自動轉成 fold_i/ckpt 形式
CKPT_DIRS = [
    Path(dir_str) / f"fold_{i}" / "ckpt"
    for i, dir_str in enumerate(TARGET_DIRS)
]

DEVICE     = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
DATA_ROOT  = "./Preprocessed_Early2Late_withLatent_0729"
warnings.filterwarnings("ignore")

# ---------- 1. 找到 best checkpoint ------------------------------------
def find_best_ckpt(ckpt_dir: Path) -> Path:
    """
    1. 優先找 best.pth
    2. 若不存在，找所有 *.pth 中有 _psnrXX.XX 格式的檔案，取 PSNR 最大的
    3. 若也沒有，退回第一個 *.pth
    """
    # 1) best.pth
    bp = ckpt_dir / "best.pth"
    if bp.is_file():
        return bp.resolve()

    # 2) 找所有 .pth
    allp = list(ckpt_dir.glob("*.pth"))
    if not allp:
        raise FileNotFoundError(f"No .pth files under {ckpt_dir}")

    # 3) 過濾出 name 中含 "_psnr" 的
    pat = re.compile(r"_psnr([0-9.]+)")
    cands = [p for p in allp if pat.search(p.stem)]
    if cands:
        # 選 psnr 最大
        best = max(cands, key=lambda p: float(pat.search(p.stem).group(1)))
        return best.resolve()

    # 4) fallback: 任選一個
    return allp[0].resolve()


# ---------- 2. DataLoader ------------------------------------------------
def make_loader(fold: int) -> DataLoader:
    ds = Early2LateWithLatentDataset(
        pet_dir=DATA_ROOT,
        dataset_type="test",
        fold=fold,
        resize_image_size=(128,128),
        split=split_all
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True)

# ---------- 3. 模型載入 --------------------------------------------------
def load_model(ckpt_path: Path):
    model = build_lkmunet_sde_film(cfg, strategy="auto").to(DEVICE)
    apply_gradient_checkpoint(model)
    st = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(st["model"], strict=True)
    model.eval()
    return model

# ---------- 4. 輔助函式 --------------------------------------------------
def ensure_5d(t):
    if t is None: return None
    while t.ndim > 5 and 1 in t.shape[1:]: t = t.squeeze(1)
    if t.ndim == 5: return t
    if t.ndim == 4: return t.unsqueeze(1)
    raise ValueError(f"latent shape {t.shape}")

def squeeze_early(x):
    return x.squeeze(2) if (x.ndim == 5 and x.size(2) == 1) else x

def pick(batch, *keys):
    for k in keys:
        if k in batch and batch[k] is not None:
            return batch[k]
    return None

# ---------- 5. 推論 & 收集 subject-wise metrics -------------------------
# 在 subject_metrics 裡新增 mse list
subject_metrics: Dict[str, Dict[str, List[float]]] = {}

for fold, ckpt_dir in enumerate(CKPT_DIRS):
    best_ckpt = find_best_ckpt(ckpt_dir)
    print(Fore.GREEN + f"[Fold {fold}] load {best_ckpt}" + Style.RESET_ALL)
    model  = load_model(best_ckpt)
    loader = make_loader(fold)

    lpips_fn = LPIPS(net="vgg").to(DEVICE).eval()
    have_gt  = "ground_truth" in loader.dataset[0]

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Fold{fold} inf"):
            subs  = batch["subject"]
            early = squeeze_early(pick(batch,"input","early")).to(DEVICE)
            #mid   = ensure_5d(pick(batch,"latent_target","mid","guide"))
            mid = None
            if mid is not None: mid = mid.to(DEVICE)
            preds = model(early, t=torch.tensor(1.0,device=DEVICE), mid_pet=mid)[0].clamp(0,1)

            for i, sid in enumerate(subs):
                p = preds[i]  # [C,H,W]
                e = early[i]
                # LPIPS 用 3-channel 輸入
                lp = lpips_fn(
                    p.unsqueeze(0).expand(1,3,*p.shape[-2:]),
                    e.unsqueeze(0).expand(1,3,*e.shape[-2:])
                ).item()

                if have_gt:
                    gt_raw = batch["ground_truth"]
                    gt     = squeeze_early(gt_raw).to(DEVICE)[i]  # [1,H,W]

                    # 計算 PSNR／SSIM
                    ps = psnr_fn(p.unsqueeze(0), gt.unsqueeze(0), data_range=1.).item()
                    ss = ssim_fn(p.unsqueeze(0), gt.unsqueeze(0), data_range=1.).item()

                    # 計算 MSE
                    # 先確保 p, gt 同 shape 且 dtype
                    mse = F.mse_loss(p, gt, reduction='mean').item()
                else:
                    ps = ss = mse = float("nan")

                # 初始化時加入 mse list
                subject_metrics.setdefault(sid, {
                    "psnr": [], "ssim": [], "lpips": [], "mse": []
                })
                subject_metrics[sid]["psnr"].append(ps)
                subject_metrics[sid]["ssim"].append(ss)
                subject_metrics[sid]["lpips"].append(lp)
                subject_metrics[sid]["mse"].append(mse)    # ← 新增

# ---------- 6. Mean ± Std & 顯示 ------------------------------------------
rows = []
for sid, m in subject_metrics.items():
    rows.append({
        "subject"    : sid,
        "psnr_mean"  : np.mean(m["psnr"]),
        "psnr_std"   : np.std(m["psnr"], ddof=0),
        "ssim_mean"  : np.mean(m["ssim"]),
        "ssim_std"   : np.std(m["ssim"], ddof=0),
        "lpips_mean" : np.mean(m["lpips"]),
        "lpips_std"  : np.std(m["lpips"], ddof=0),
        "mse_mean"   : np.mean(m["mse"]),          # ← 新增
        "mse_std"    : np.std(m["mse"], ddof=0),   # ← 新增
    })
df = pd.DataFrame(rows).set_index("subject")

print("\n==== Subject-wise mean ± std across folds ====")
print(df)

overall = {
    "psnr_mean" : df["psnr_mean"].mean(),
    "psnr_std"  : df["psnr_mean"].std(),
    "ssim_mean" : df["ssim_mean"].mean(),
    "ssim_std"  : df["ssim_mean"].std(),
    "lpips_mean": df["lpips_mean"].mean(),
    "lpips_std" : df["lpips_mean"].std(),
    "mse_mean"  : df["mse_mean"].mean(),        # ← 新增
    "mse_std"   : df["mse_mean"].std(),         # ← 新增
}
print("\n==== Overall across subjects ====")
print(pd.Series(overall))

# ---------- 7. 存成 CSV ------------------------------------------
df.reset_index().to_csv("inference_subject_metrics_SDE_LPSenh_0729_major_revision.csv", index=False)
print(Fore.GREEN + "→ 已儲存 inference_subject_metrics_SDE_LPSenh_0729_major_revision.csv" + Style.RESET_ALL)

pd.DataFrame([overall]).to_csv("inference_overall_metrics_SDE_LPSenh_0729_major_revision.csv", index=False)
print(Fore.GREEN + "→ 已儲存 inference_overall_metrics_SDE_LPSenh_0729_major_revision.csv" + Style.RESET_ALL)
